# Stateful decoding policies — simulation and the C1a gate

The released dataset is unreproducible: its generator sampled with `torch.multinomial` and never
called `torch.manual_seed`, so the hidden policies existed only in that process's memory and no
mode labels can be recovered. This notebook generates a fresh simulation that records everything,
then runs the one gate that decides whether the main experiment is worth building.

**The design principle.** The base model's next-token distribution is stored at every step. Every
sampler in the grammar is a deterministic function of it, so afterwards every fit, baseline,
ablation and diagnostic runs on CPU with no model and no GPU. It is the only quantity that cannot
be reconstructed without repeating this run.

**The gate.** `C1a` reveals the true mode label on the calibration traces and nowhere else, then
fits the policy with that supervision. It is exactly what a perfect mode predictor would hand
downstream, so it is the real ceiling of the amortized approach — not the true-policy oracle.

```
PASS   C1a >= 0.70 of headroom  -> build the predictor; it must then reach
                                   >= 0.60 of headroom AND >= 0.70 x C1a
FAIL   C1a <  0.70              -> stop. Revealing the modes does not identify
                                   the policies, so 0.60 is unreachable by
                                   construction and a predictor adjudicates nothing.
```

Pre-committed before any number was seen. GPU on for section 3; everything after is CPU.

## 1. Environment

In [ ]:
import torch, platform
print('torch', torch.__version__, '| python', platform.python_version())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('gpu  ', p.name, '|', round(p.total_memory / 1024**3, 1), 'GiB')
else:
    print('gpu   NONE -- generation will not finish in reasonable time')

## 2. Scripts and prompts

Self-contained: the prompt corpus is generated here from templates rather than read from any
external dataset, so this notebook needs nothing attached.

The corpus is built for spread across MODEL STATES, not literary variety. Guards fire on entropy,
on the top-two logit gap, on repetition -- if every prompt puts the model in the same state, the
same guards fire in the same places and much of the policy space never shows itself. Enumeration
prompts matter disproportionately: the repetition sensors produce no evidence at all in
non-repetitive text, so without prompts that invite repeats those parameters are unidentifiable
by absence.

In [ ]:
%%writefile /kaggle/working/grammar_v3.py
"""Real decoding mechanisms wrapped in a mode machine.

v2 kept the real mechanisms (min_p, typical, top_nsigma, mirostat, dynatemp, XTC, hidden
sampler order) but had only ~80 discrete structures. With 48 random candidates roughly five
already had the right truncation family AND order, so guessing the structure was a lottery
rather than a search: random 0.623, hill-climb 0.667, gate 0.707 -- all within 0.084.

Here each policy is a small STATE MACHINE. Every mode carries its own full sub-policy, and
transitions are guarded by conditions on sensors computed from the model's RAW distribution.
Structure count rises to ~3x10^5 even at M=2, because the guards multiply:

    M(2) x order(2) x trunc^M(25) x [sensor(5) x cmp(2)]^guards(2) x dwell(4) x expiry(16)

Two properties keep this fair rather than merely hard:

  * every sensor is computed on the RAW temperature-1 distribution, so it cannot see the
    policy's own parameters. A sensor threshold can therefore never trade off against a
    temperature, which is what created the m^T degeneracy in the first design.
  * all state is a deterministic function of the OBSERVED tokens. Replaying a trace under a
    candidate gives an exact likelihood in one pass -- no filtering over hidden paths.

Calibration is raised to 8 traces, because v2's gate reached only 0.707: with the structure
handed over it still could not pin the numbers, which is evidence-limited, and the fix for
that is more evidence.

    NTASK=3 python grammar_v3.py
"""
import os, sys, math, random, copy
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = os.environ.get('MODEL', 'Qwen/Qwen2.5-0.5B')
NTASK = int(os.environ.get('NTASK', 8))
NCAL = int(os.environ.get('NCAL', 8))
CALLEN = int(os.environ.get('CALLEN', 32))
NQ = int(os.environ.get('NQ', 40))
KCAND = int(os.environ.get('KCAND', 64))
HILL = int(os.environ.get('HILL', 150))
GATE = int(os.environ.get('GATE', 400))
EPS = float(os.environ.get('EPS', 1e-3))
SEED = 71294
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

OPENERS = ["The main reason", "In the morning", "She decided to", "One problem with",
           "After the meeting", "The difference between", "A common mistake", "He looked at",
           "The report said", "Most people think", "It turns out", "The city of",
           "During the winter", "They agreed that", "The first step", "Nobody expected"]
TOPICS = ["the weather", "a broken engine", "her new job", "the missing key", "local farming",
          "a long journey", "the price of bread", "an old photograph", "a difficult exam",
          "the train schedule", "the harvest", "a strange noise", "the election", "a lost letter"]

TRUNCS = ['none', 'min_p', 'typical', 'top_nsigma', 'mirostat2']
ORDERS = ['llama', 'hf']
GAMMAS = [0.60, 0.80, 0.90, 0.97]
SENSORS = ['H', 'G', 'U', 'D', 'P']
SENSOR_RANGE = dict(H=(0.5, 6.0), G=(0.2, 8.0), U=(0.5, 9.0), D=(4, 16), P=(6, 34))
TP_BOUNDS = {'none': (0.0, 1.0), 'min_p': (0.01, 0.35), 'typical': (0.15, 0.99),
             'top_nsigma': (0.4, 3.0), 'mirostat2': (2.0, 9.0)}
BOUNDS = dict(eta=(0.01, 0.35), t_lo=(0.30, 1.00), t_hi=(0.90, 1.80), t_exp=(0.3, 3.0),
              xtc_q=(0.0, 0.7), xtc_t=(0.03, 0.30), lam=(0.0, 3.5))


def sample_mode(rng):
    tr = rng.choice(TRUNCS)
    lo, hi = TP_BOUNDS[tr]
    return dict(trunc=tr, tp=round(rng.uniform(lo, hi), 4),
                eta=round(rng.uniform(0.02, 0.30), 4),
                dyn=rng.random() < 0.35,
                t_lo=round(rng.uniform(0.35, 0.95), 3),
                t_hi=round(rng.uniform(0.90, 1.70), 3),
                t_exp=round(rng.uniform(0.5, 2.5), 3),
                xtc_q=0.0 if rng.random() < 0.5 else round(rng.uniform(0.10, 0.60), 3),
                xtc_t=round(rng.uniform(0.05, 0.25), 3),
                lam=0.0 if rng.random() < 0.4 else round(rng.uniform(0.4, 3.0), 3))


def sample_policy(rng):
    M = 2 if rng.random() < 0.6 else 3
    pairs = [(0, 1), (1, 0)] if M == 2 else [(0, 1), (1, 2), (1, 0), (2, 0)]
    guards = []
    for src, dst in pairs:
        s = rng.choice(SENSORS)
        lo, hi = SENSOR_RANGE[s]
        guards.append(dict(src=src, dst=dst, sensor=s, cmp=rng.choice(['ge', 'lt']),
                           thr=round(rng.uniform(lo, hi), 3)))
    return dict(M=M, order=rng.choice(ORDERS), gamma=rng.choice(GAMMAS),
                modes=[sample_mode(rng) for _ in range(M)], guards=guards,
                dwell=[rng.choice([0, 3]) for _ in range(M)],
                expiry=[rng.choice([8, 16, 32, 999]) for _ in range(M)])


def active_keys(md):
    ks = ['tp'] if md['trunc'] != 'none' else []
    if md['trunc'] == 'mirostat2':
        ks.append('eta')
    ks.append('t_hi')
    if md['dyn']:
        ks += ['t_lo', 't_exp']
    if md['xtc_q'] > 0:
        ks += ['xtc_q', 'xtc_t']
    if md['lam'] > 0:
        ks.append('lam')
    return ks


def bounds_for(md, k):
    return TP_BOUNDS[md['trunc']] if k == 'tp' else BOUNDS[k]


# ---------------------------------------------------------------- sensors and the machine

def sensors(p_raw, z, st, pos):
    """Computed on the RAW distribution only, so they never see policy parameters."""
    lp = torch.log(p_raw.clamp_min(1e-12))
    H = float(-(p_raw * lp).sum())
    top2 = torch.topk(z.float(), 2).values
    G = float(top2[0] - top2[1])
    hist = st['hist'][-16:]
    return dict(H=H, G=G, U=st['prev_surprisal'], D=float(len(set(hist))),
                P=float(pos))


def fsm_step(pol, st, s):
    m, c = st['m'], st['c']
    if c < pol['dwell'][m]:
        st['c'] = c + 1
        return m
    if m != 0 and c >= pol['expiry'][m]:
        st['m'], st['c'] = 0, 0
        return 0
    for g in pol['guards']:
        if g['src'] != m:
            continue
        v = s[g['sensor']]
        if (v >= g['thr']) if g['cmp'] == 'ge' else (v < g['thr']):
            st['m'], st['c'] = g['dst'], 0
            return g['dst']
    st['c'] = c + 1
    return m


# ---------------------------------------------------------------- the per-token transform

def temperature(md, H):
    if not md['dyn']:
        return md['t_hi']
    return md['t_lo'] + (md['t_hi'] - md['t_lo']) * (max(H, 0) / 11.93) ** md['t_exp']


def truncate(p, md, mu):
    tr, v = md['trunc'], md['tp']
    if tr == 'none':
        return torch.ones_like(p, dtype=torch.bool)
    if tr == 'min_p':
        return p >= p.max() * v
    if tr == 'typical':
        lp = torch.log(p.clamp_min(1e-12))
        ent = -(p * lp).sum()
        order = (-lp - ent).abs().argsort()
        keep = (p[order].cumsum(0) - p[order]) < v
        m = torch.zeros_like(p, dtype=torch.bool)
        m[order] = keep
        m[p.argmax()] = True
        return m
    if tr == 'top_nsigma':
        zz = torch.log(p.clamp_min(1e-30))
        return zz >= zz.max() - v * zz.std()
    return p >= 2.0 ** (-mu)


def xtc_mask(p, thr):
    B = p >= thr
    if int(B.sum()) < 2:
        return None
    idx = torch.nonzero(B).squeeze(1)
    keep = idx[p[idx].argmin()]
    m = torch.ones_like(p, dtype=torch.bool)
    m[idx] = False
    m[keep] = True
    return m


def policy_dist(z, pol, st, pos):
    p_raw = F.softmax(z.float(), dim=-1)
    s = sensors(p_raw, z, st, pos)
    m = fsm_step(pol, st, s)
    md = pol['modes'][m]

    lg = z.float().clone()
    if md['lam'] > 0 and st['u']:
        idx = torch.tensor(list(st['u'].keys()), device=lg.device, dtype=torch.long)
        val = torch.tensor([min(v, 3.0) for v in st['u'].values()], device=lg.device)
        lg[idx] = lg[idx] - md['lam'] * val
    base = F.softmax(lg, dim=-1)
    T = temperature(md, s['H'])

    def finish(b0, temp_after):
        msk = truncate(b0, md, st['mu'])
        b = b0 * msk
        if float(b.sum()) <= 0:
            b = b0.clone()
        branches = [(1.0, b)]
        if md['xtc_q'] > 0:
            xm = xtc_mask(b / b.sum(), md['xtc_t'])
            if xm is not None and float((b * xm).sum()) > 0:
                branches = [(1 - md['xtc_q'], b), (md['xtc_q'], b * xm)]
        out = torch.zeros_like(b0)
        for w, br in branches:
            q = br / br.sum().clamp_min(1e-30)
            if temp_after:
                q = q.clamp_min(1e-30) ** (1.0 / T)
                q = q / q.sum()
            out += w * q
        return out

    if pol['order'] == 'llama':
        return finish(base, True), p_raw, md
    return finish(F.softmax(lg / max(T, 1e-3), dim=-1), False), p_raw, md


def init_state(pol):
    md = pol['modes'][0]
    return dict(m=0, c=0, u={}, hist=[], prev_surprisal=0.0,
                mu=2.0 * md['tp'] if md['trunc'] == 'mirostat2' else 5.0)


def advance(st, pol, md, p_raw, tok):
    S = -math.log2(max(float(p_raw[tok]), 1e-12))
    if md['trunc'] == 'mirostat2':
        st['mu'] = float(np.clip(st['mu'] - md['eta'] * (S - md['tp']), 0.5, 20.0))
    if md['lam'] > 0:
        g = pol['gamma']
        st['u'] = {k: v * g for k, v in st['u'].items() if v * g > 1e-3}
        st['u'][tok] = st['u'].get(tok, 0.0) + g
    st['prev_surprisal'] = S * math.log(2)
    st['hist'].append(tok)


# ---------------------------------------------------------------- data and fitting

@torch.no_grad()
def build_task(model, tok, rng, tries=6):
    """Resample any policy whose modes are not both genuinely occupied.

    If the guards rarely fire the machine sits in mode 0 forever, the other mode's parameters
    are never exercised, and no method can identify them. That is unfair rather than hard, so
    those policies must never reach the dataset."""
    for attempt in range(tries):
        true = sample_policy(rng)
        cal, qry, occ = [], [], []
        for tr in range(NCAL):
            ids = torch.tensor([tok.encode('%s %s' % (rng.choice(OPENERS),
                                                      rng.choice(TOPICS)))], device=DEV)
            st = init_state(true)
            for pos in range(CALLEN):
                z = model(input_ids=ids).logits[0, -1].half()
                d, praw, md = policy_dist(z, true, st, pos)
                nxt = int(torch.multinomial(d, 1))
                cal.append((tr, z, pos, nxt))
                occ.append(st['m'])
                advance(st, true, md, praw, nxt)
                ids = torch.cat([ids, torch.tensor([[nxt]], device=DEV)], dim=1)
        share = [occ.count(m) / len(occ) for m in range(true['M'])]
        if min(share) >= 0.08:
            break
        print('    resample: mode occupancy %s' % ['%.2f' % s for s in share], flush=True)
    true['_occ'] = share
    ids = torch.tensor([tok.encode('%s %s' % (rng.choice(OPENERS), rng.choice(TOPICS)))],
                       device=DEV)
    st = init_state(true)
    # Every position from 0 is kept, with a flag for whether it is scored. The burn-in must be
    # replayed, not skipped: mode, mirostat mu, the decay field and the token history are all
    # functions of the prefix, so a candidate started fresh at position 10 evaluates every
    # scored position with the wrong state. Query contexts MUST carry their full prefix.
    for pos in range(NQ + 10):
        z = model(input_ids=ids).logits[0, -1].half()
        d, praw, md = policy_dist(z, true, st, pos)
        nxt = int(torch.multinomial(d, 1))
        qry.append((z, pos, nxt, float(d[nxt]), float(praw[nxt]), pos >= 10))
        advance(st, true, md, praw, nxt)
        ids = torch.cat([ids, torch.tensor([[nxt]], device=DEV)], dim=1)
    return true, cal, qry


def loglik(pol, cal):
    s, st, last = 0.0, None, -1
    for tr, z, pos, nxt in cal:
        if tr != last:
            st, last = init_state(pol), tr
        d, praw, md = policy_dist(z, pol, st, pos)
        s += math.log(max(float(d[nxt]), 1e-12))
        advance(st, pol, md, praw, nxt)
    return s


def predict(pol, qry):
    """Replay the whole generation; return predictions for the scored positions only."""
    out, st = [], init_state(pol)
    for z, pos, nxt, po, pb, scored in qry:
        d, praw, md = policy_dist(z, pol, st, pos)
        if scored:
            out.append(float(d[nxt]))
        advance(st, pol, md, praw, nxt)
    return out


def coord_list(pol):
    """Every fittable continuous coordinate, as (kind, index, key)."""
    out = []
    for i, md in enumerate(pol['modes']):
        out += [('mode', i, k) for k in active_keys(md)]
    out += [('guard', j, 'thr') for j in range(len(pol['guards']))]
    return out


def jitter(pol, rng, scale, keys_only=False, ncoord=1):
    """Move a FEW coordinates, not all of them.

    Perturbing 10-15 dimensions at once makes acceptance rare and the climb stall -- that is
    why the first version scored below random search."""
    p = copy.deepcopy(pol)
    coords = coord_list(p)
    for kind, i, k in rng.sample(coords, min(ncoord, len(coords))):
        if kind == 'mode':
            md = p['modes'][i]
            lo, hi = bounds_for(md, k)
            md[k] = float(np.clip(md[k] + rng.gauss(0, scale * (hi - lo)), lo, hi))
        else:
            g = p['guards'][i]
            lo, hi = SENSOR_RANGE[g['sensor']]
            g['thr'] = float(np.clip(g['thr'] + rng.gauss(0, scale * (hi - lo)), lo, hi))
    if not keys_only and rng.random() < 0.2:
        p['order'] = rng.choice(ORDERS)
    if not keys_only and rng.random() < 0.2:
        md = p['modes'][rng.randrange(p['M'])]
        md['trunc'] = rng.choice(TRUNCS)
        lo, hi = TP_BOUNDS[md['trunc']]
        md['tp'] = rng.uniform(lo, hi)
    if not keys_only and p['guards'] and rng.random() < 0.2:
        g = p['guards'][rng.randrange(len(p['guards']))]
        g['sensor'] = rng.choice(SENSORS)
        lo, hi = SENSOR_RANGE[g['sensor']]
        g['thr'] = rng.uniform(lo, hi)
        g['cmp'] = rng.choice(['ge', 'lt'])
    return p


def loss(p, pb):
    return -math.log(max((1 - EPS) * p + EPS * pb, 1e-12))


def main():
    tok = AutoTokenizer.from_pretrained(MODEL)
    dt = torch.float32 if DEV == 'cpu' else torch.bfloat16
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dt).to(DEV).eval()
    rng = random.Random(SEED)
    print('tasks=%d calibration=%dx%d queries=%d candidates=%d hill=%d gate=%d'
          % (NTASK, NCAL, CALLEN, NQ, KCAND, HILL, GATE), flush=True)

    tot = dict(marg=0.0, orc=0.0, rand=0.0, hill=0.0, gate=0.0, n=0)
    for t in range(NTASK):
        true, cal, qry = build_task(model, tok, rng)
        cands = [sample_policy(rng) for _ in range(KCAND)]
        cols = [predict(c, qry) for c in cands]
        ll = np.array([loglik(c, cal) for c in cands])
        pprior = [float(np.mean(col)) for col in zip(*cols)]
        w = np.exp(ll - ll.max()); w /= w.sum()
        pmix = [float(np.dot(w, col)) for col in zip(*cols)]

        cur, cur_ll = copy.deepcopy(cands[int(np.argmax(ll))]), float(ll.max())
        for _ in range(HILL):
            prop = jitter(cur, rng, 0.12, ncoord=2)
            pl = loglik(prop, cal)
            if pl > cur_ll:
                cur, cur_ll = prop, pl

        gate = jitter(true, rng, 0.60, keys_only=True, ncoord=99)   # randomise the numbers
        g_ll = loglik(gate, cal)
        for _ in range(GATE):
            prop = jitter(gate, rng, 0.15, keys_only=True, ncoord=1)  # then coordinate-wise
            pl = loglik(prop, cal)
            if pl > g_ll:
                gate, g_ll = prop, pl

        sc = [q for q in qry if q[5]]
        for name, preds in [('rand', pmix), ('hill', predict(cur, qry)),
                            ('gate', predict(gate, qry))]:
            tot[name] += sum(loss(p, q[4]) for p, q in zip(preds, sc))
        tot['marg'] += sum(loss(p, q[4]) for p, q in zip(pprior, sc))
        tot['orc'] += sum(loss(q[3], q[4]) for q in sc)
        tot['n'] += len(sc)
        print('  task %d/%d M=%d order=%s truncs=%s' %
              (t + 1, NTASK, true['M'], true['order'],
               '/'.join(m['trunc'] for m in true['modes'])), flush=True)

    m, o, den = tot['marg'], tot['orc'], tot['marg'] - tot['orc']
    print('\n' + '=' * 68)
    print('GRAMMAR V3 -- mode machine over real mechanisms (%d positions)' % tot['n'])
    print('=' * 68)
    print('  marginal %.4f | oracle %.4f | headroom %.4f nats/pos'
          % (m / tot['n'], o / tot['n'], den / tot['n']))
    print('\n  %-38s %10s' % ('method', 'headroom captured'))
    for k, lab in [('rand', 'random search + posterior'),
                   ('hill', 'hill-climb (%d proposals)' % HILL),
                   ('gate', 'GATE: structure known')]:
        print('  %-38s %10.3f' % (lab, (m - tot[k]) / den if den else 0.0))
    print('\n  want: gate >= 0.85, random and hill far below it.')
    print('  gate low  -> evidence-limited, raise NCAL. gate high and the others low')
    print('  -> search-limited, which is the design working.')


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /kaggle/working/make_prompts.py
"""Build the prompt corpus.

The pilot used 40 openers crossed with 18 topics -- 720 strings, often ungrammatical, all in
one register. Prompts are drawn independently of the hidden policy, so prompt identity leaks
nothing about the answer. What a narrow corpus does harm is the SENSOR distribution: guards
fire on entropy, on the top-two logit gap, on repetition. If every prompt puts the model in the
same state, the same guards fire in the same places and much of the policy space never shows
itself.

So the corpus is built for spread across model states rather than literary variety:

  narrative      open continuation, high entropy
  factual        encyclopedic framing, the model commits harder
  instructional  imperative, formulaic continuations
  dialogue       quoted speech, distinct token statistics
  enumeration    lists. These matter disproportionately: the repetition field and the
                 distinct-token sensor produce NO evidence in non-repetitive text, so without
                 prompts that invite repeats those parameters are unidentifiable by absence.
  technical      jargon, low-entropy stretches
  question       interrogative, mid entropy
  temporal       dated openings, strong priors

Registers are drawn to an equal quota. Sampling them at random exhausts the smaller ones first
and leaves the remainder dominated by whichever register has the largest combination count --
a first attempt produced a query set that was 100% enumeration.

Calibration and query prompts are disjoint, so per-prompt behaviour cannot be memorised from
the traces and reused at scored positions.

    python make_prompts.py
    NCAL=20000 NQUERY=10000 python make_prompts.py
"""
import os, csv, random
from collections import Counter

NCAL = int(os.environ.get('NCAL', 20000))
NQUERY = int(os.environ.get('NQUERY', 10000))
OUT = os.environ.get('OUT', '.')
SEED = 71294

PERSON = ['she', 'he', 'they', 'the driver', 'the nurse', 'the teacher', 'the engineer',
          'the farmer', 'the clerk', 'my brother', 'the manager', 'the pilot', 'the tenant',
          'the inspector', 'the courier', 'the gardener', 'the neighbour', 'the referee',
          'the plumber', 'the archivist', 'the barista', 'the surveyor', 'the porter',
          'the auditor', 'the curator', 'the welder', 'the midwife', 'the sailor',
          'the printer', 'the cook', 'the guard', 'the miller', 'the tailor', 'the vet',
          'the warden', 'the cellist', 'the glazier', 'the mason', 'the skipper']
OBJECT = ['engine', 'letter', 'ledger', 'kettle', 'ladder', 'switch', 'bridge', 'harvest',
          'invoice', 'passport', 'freezer', 'antenna', 'contract', 'blanket', 'compass',
          'furnace', 'suitcase', 'gearbox', 'notebook', 'thermostat', 'radiator', 'cistern',
          'railing', 'awning', 'turbine', 'palette', 'kiln', 'trellis', 'weir', 'flue',
          'lantern', 'anchor', 'satchel', 'pulley', 'cabinet', 'hinge', 'canopy', 'stirrup',
          'crate', 'valve', 'bellows', 'mallet', 'sieve', 'trolley', 'beacon', 'quilt',
          'harness', 'chisel', 'lattice', 'spindle']
PLACE = ['Lisbon', 'Dundee', 'Nairobi', 'Osaka', 'Cordoba', 'Halifax', 'Bergen', 'Perth',
         'Cusco', 'Tromso', 'Ghent', 'Split', 'Antwerp', 'Kanazawa', 'Rovaniemi', 'Cadiz',
         'Trieste', 'Aarhus', 'Galway', 'Utrecht', 'Salerno', 'Bruges', 'Tartu', 'Nara',
         'the valley', 'the coast', 'the border', 'the marshes', 'the highlands',
         'the estuary', 'the delta', 'the plateau', 'the foothills']
ACTION = ['stopped', 'refused', 'waited', 'agreed', 'argued', 'left', 'returned', 'complained',
          'apologised', 'insisted', 'hesitated', 'relented', 'objected', 'volunteered',
          'listened', 'answered', 'withdrew', 'conceded', 'protested', 'signed', 'resigned',
          'reconsidered', 'intervened', 'declined']
FIELD = ['drainage', 'masonry', 'grammar', 'accounting', 'forestry', 'cartography', 'welding',
         'typography', 'irrigation', 'metallurgy', 'ornithology', 'ceramics', 'hydrology',
         'bookbinding', 'horticulture', 'seismology', 'lexicography', 'glassblowing',
         'navigation', 'apiculture', 'joinery', 'thatching', 'toponymy', 'viticulture']
TECH = ['parser', 'buffer', 'compiler', 'scheduler', 'cache', 'index', 'daemon', 'socket',
        'checksum', 'allocator', 'iterator', 'mutex', 'registry', 'codec', 'kernel',
        'linker', 'profiler', 'dispatcher', 'validator', 'serialiser', 'router', 'tokeniser',
        'collector', 'accumulator', 'watchdog', 'sandbox']
ADJ = ['broken', 'missing', 'frozen', 'unmarked', 'faulty', 'spare', 'rusted', 'sealed',
       'temporary', 'unused', 'damp', 'narrow', 'crowded', 'quiet', 'expensive', 'cracked',
       'borrowed', 'painted', 'folded', 'loose', 'heavy', 'shallow', 'bent', 'dusty']
YEAR = [str(y) for y in range(1861, 2021, 3)]
# bare, so "Every {season}" and "By {season}" read correctly; an article-bearing entry like
# "the monsoon" produced "Every the monsoon"
SEASON = ['spring', 'summer', 'autumn', 'winter', 'harvest', 'midwinter', 'midsummer']

REGISTERS = {
    'narrative':     ["After the {adj} {obj}, {per}", "When {per} finally reached the {obj}",
                      "{per} had already sold the {obj}", "Before the {adj} {obj} arrived, {per}",
                      "Once {per} noticed the {adj} {obj}", "{per} {act} near the {obj}"],
    'factual':       ["The {field} of {place} is", "{place} is known for its {obj}",
                      "The {adj} {obj} of {place} was", "Records from {place} list the {obj}",
                      "Most {field} in {place} depends on"],
    'instructional': ["To repair the {adj} {obj}, first", "Never leave the {adj} {obj} near",
                      "Check the {obj} before you", "First remove the {adj} {obj}",
                      "Always store the {obj} beside the {obj}",
                      "Do not open the {adj} {obj} until", "Replace the {obj} whenever the {obj}",
                      "Ask {per} before moving the {obj}", "Label every {obj} in the {place}",
                      "Keep the {adj} {obj} away from"],
    'dialogue':      ["\"I never {act},\" said {per}", "\"The {obj} is {adj},\" {per} replied",
                      "{per} asked whether the {obj} was", "\"Tell me about the {field},\" said {per}",
                      "\"Where is the {adj} {obj}?\" asked {per}"],
    'enumeration':   ["The list included {obj}, {obj}, and", "Items: {obj}, {obj}, {obj}, and",
                      "We packed a {obj}, a {obj}, and", "First the {obj}, then the {obj}, then",
                      "{obj}, {obj}, {obj}, and then"],
    'technical':     ["The {tech} returns a {adj}", "If the {tech} fails, the {tech}",
                      "Each {tech} holds a {adj} {obj}", "The {tech} used in {field} must",
                      "Every {tech} writes to the {tech}"],
    'question':      ["Why did {per} {act} near the {obj}?", "What happens when the {adj} {obj}",
                      "How is the {field} of {place}", "Who owns the {adj} {obj} in {place}?",
                      "When did {per} last {act}?"],
    'temporal':      ["In {year}, the {adj} {obj} was", "By {season}, {per} had",
                      "Since {year}, {place} has kept", "Every {season} the {obj} is",
                      "Until {year}, the {obj} in {place}"],
}

SLOTS = dict(per=PERSON, obj=OBJECT, place=PLACE, act=ACTION, field=FIELD,
             tech=TECH, adj=ADJ, year=YEAR, season=SEASON)


def fill(pattern, rng):
    out = pattern
    for key, vocab in SLOTS.items():
        token = '{%s}' % key
        while token in out:
            out = out.replace(token, rng.choice(vocab), 1)
    # capitalise the first letter, skipping a leading quote
    for i, ch in enumerate(out):
        if ch.isalpha():
            return out[:i] + ch.upper() + out[i + 1:]
    return out


def register_space(reg):
    total = 0
    for p in REGISTERS[reg]:
        n = 1
        for key, vocab in SLOTS.items():
            n *= len(vocab) ** p.count('{%s}' % key)
        total += n
    return total


def build(n, rng, seen):
    """Equal quota per register; random sampling exhausts small registers and skews the rest."""
    regs = sorted(REGISTERS)
    quota = {r: n // len(regs) for r in regs}
    for r in regs[:n % len(regs)]:
        quota[r] += 1
    out = []
    for r in regs:
        got, tries = 0, 0
        while got < quota[r] and tries < quota[r] * 200:
            tries += 1
            s = fill(rng.choice(REGISTERS[r]), rng)
            if s in seen:
                continue
            seen.add(s)
            out.append((r, s))
            got += 1
        if got < quota[r]:
            print('  WARNING: %s produced %d of %d requested' % (r, got, quota[r]))
    rng.shuffle(out)
    return out


def write(path, rows, prefix):
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['prompt_id', 'text'])
        for i, (_, s) in enumerate(rows):
            w.writerow(['%s%06d' % (prefix, i), s])
    # register recorded separately: recovering it from the text by prefix is unreliable
    # (`The ` opens several registers), and it is an internal label, not part of the released
    # column spec
    with open(path.replace('.csv', '_registers.csv'), 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['prompt_id', 'register'])
        for i, (r, _) in enumerate(rows):
            w.writerow(['%s%06d' % (prefix, i), r])


def main():
    rng = random.Random(SEED)
    print('combinations per register:')
    for r in sorted(REGISTERS):
        print('  %-14s %.3g' % (r, register_space(r)))
    need = (NCAL + NQUERY) // len(REGISTERS)
    print('needed per register: %d\n' % need)

    seen = set()
    cal = build(NCAL, rng, seen)
    qry = build(NQUERY, rng, seen)
    assert not ({s for _, s in cal} & {s for _, s in qry}), 'calibration and query overlap'

    write(os.path.join(OUT, 'prompts_calibration.csv'), cal, 'PC')
    write(os.path.join(OUT, 'prompts_query.csv'), qry, 'PQ')

    print('calibration %d | query %d | disjoint: yes' % (len(cal), len(qry)))
    for name, rows in [('calibration', cal), ('query', qry)]:
        c = Counter(r for r, _ in rows)
        wl = [len(s.split()) for _, s in rows]
        print('  %-12s words %d-%d mean %.1f | registers %s'
              % (name, min(wl), max(wl), sum(wl) / len(wl),
                 ' '.join('%s:%d' % (k, v) for k, v in sorted(c.items()))))
    print('\nsamples:')
    for r, s in cal[:12]:
        print('  %-14s %s' % (r, s))


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /kaggle/working/probe.py
"""The probe: how a scored item is built, and the prompt loader.

Extracted so the research code carries no challenge-specific machinery -- no splitting, no
grading, no answer files.

MATCHED-STRATUM CONSTRUCTION. The candidate set is built FIRST, from the base model alone and
without reference to the policy: a window of adjacent base-model ranks, from which K distinct
tokens are drawn uniformly. Because the window is narrow every candidate carries almost the same
base probability, so base rank cannot identify the label. Only then is the label drawn, from the
policy restricted to those candidates.

The alternative -- sampling decoys from a flattened prior -- fails structurally: decoys drawn
from a flatter distribution are rarer tokens, so the emitted one is simply whichever candidate
the base model likes most. Measured, a baseline using the weights but nothing about the policy
scored 0.981, with the emitted token being the base argmax 81.3% of the time against 12.5%
chance. Every flattening setting traded that shortcut against duplicate candidates and vanishing
signal.
"""
import os, csv, math
import numpy as np
import torch

import grammar_v3 as G

KCAND = int(os.environ.get('KCAND', 8))          # candidates per scored item
STRATUM_TOP = int(os.environ.get('STRATUM_TOP', 512))
STRATUM_W = int(os.environ.get('STRATUM_W', 384))
# items whose reference distribution is near-uniform carry almost no information. The test is on
# the candidate SET -- the largest reference probability across candidates -- never on which
# label was drawn: filtering on the drawn label would be outcome-dependent and would bias the
# surviving labels toward likely candidates, which a solver could exploit by sharpening.
MIN_MAXP = float(os.environ.get('MIN_MAXP', 0.18))


def load_prompts(path):
    return [(r['prompt_id'], r['text']) for r in csv.DictReader(open(path, encoding='utf-8'))]


def scored_item(z_b, pol, st_b, step, rng):
    """Build one scored item at the position reached by a prefix.

    MATCHED STRATUM construction. The candidate set is built FIRST, from the base model alone
    and without reference to the policy: a window of adjacent base-model ranks, from which K
    distinct tokens are drawn uniformly. Because the window is narrow, every candidate has
    almost the same base probability. Only then is the label drawn, from the policy restricted
    to those candidates.

    This replaces sampling decoys from a flattened prior mixture, which failed structurally.
    Decoys drawn from a flatter distribution than the policy are rarer tokens, so the emitted
    one was simply whichever candidate the base model liked most -- a baseline using the
    weights but nothing about the policy scored 0.981, with the emitted token being the base
    argmax 81.3% of the time against 12.5% chance. Every setting of the flattening parameter
    traded that shortcut against duplicate candidates and vanishing signal.

    Here there is no decoy distribution to estimate, no duplicates by construction, and base
    rank cannot identify the label because the candidates are matched on base probability. What
    is left to infer is how the policy redistributes mass within the set.
    """
    q, p_raw, _ = G.policy_dist(z_b, pol, st_b, step)

    top = torch.topk(p_raw.float(), STRATUM_TOP)
    # a log-uniform anchor: narrow windows near the head, wider spacing down the tail, which is
    # where truncation boundaries actually fall
    hi = STRATUM_TOP - STRATUM_W
    a = int(math.exp(rng.uniform(0.0, math.log(hi))))
    a = max(0, min(a, hi - 1))
    window = top.indices[a:a + STRATUM_W]

    pick = torch.randperm(STRATUM_W, generator=torch.Generator().manual_seed(rng.randrange(1 << 30)))
    cands = window[pick[:KCAND]].tolist()
    if len(set(cands)) < KCAND:
        return None

    ct = torch.tensor(cands, device=z_b.device)
    w = q[ct].double()
    if float(w.sum()) <= 0:
        return None                      # the policy truncates the whole window away
    p = (w / w.sum()).cpu().numpy()

    idx = int(np.searchsorted(np.cumsum(p), rng.random()))
    idx = min(idx, KCAND - 1)

    # the base model restricted to the same candidates. This is the zero point of the metric:
    # nearly every mechanism in the grammar is a MONOTONE transform of the base distribution --
    # temperature preserves order, truncation removes a tail without reordering what is left --
    # so the candidate the policy most likely emitted is usually the one the base model already
    # ranked highest. Measured, base ranking captured 0.72-0.80 of the uniform-normalised score
    # at every stratum width tried. Scoring against uniform would hand that away for free;
    # scoring against the base model asks the question that matters, which is what knowing the
    # POLICY adds beyond knowing the model.
    b = p_raw[ct].double()
    b = (b / b.sum().clamp_min(1e-30)).cpu().numpy()
    base_spread = float((top.values[a] / top.values[a + STRATUM_W - 1]).log())
    return cands, idx, float(p[idx]), float(b[idx]), float(p.max()), base_spread

In [ ]:
%%writefile /kaggle/working/policy_cpu.py
"""Re-implementation of the policy transform over the stored top-K support, in numpy.

Every sampler in grammar_v3 is a deterministic function of the base distribution and its
parameters. simlab.py stores that distribution at every step, so the whole fitting problem runs
on CPU here with no model and no GPU. This module is the bridge.

Correctness note: candidates are drawn from the top STRATUM_TOP=512 base ranks, which is exactly
the stored support, so no candidate is ever outside it. The one approximation is the repetition
penalty, which in the generator applies to any token in the recent history; here it applies only
to history tokens that fall inside the stored top-K. verify_against_generator() measures that gap
rather than assuming it away.
"""
import numpy as np

# Bump on every change. The audit prints it, so a stale import is visible instead of silent --
# %%writefile replaces the FILE, but an already-imported module stays in memory, and we lost
# three audit runs to that before making the version observable.
VERSION = '2026-10-09.mirostat-state-fix'

SENSORS = ['H', 'G', 'U', 'D', 'P']
_TYPICAL_ENT = [0.0]   # stored full entropy, tempered where the sampler sees it tempered
_ZSTD = [None]         # std of full-vocab log-probs, scaled by 1/T for hf order


def grid_at(T, tgrid, vals):
    """Interpolate a stored temperature grid QUADRATICALLY in log-T.

    logsumexp(z/T) and the tempered entropy are smooth in log-T, and the grid is geometric, so
    quadratic interpolation is 7x more accurate than linear on the same points: max error drops
    from 1.1e-2 to 1.6e-3 nats. Free, and needs no regeneration.
    """
    tg = np.asarray(tgrid, dtype=np.float64)
    v = np.asarray(vals, dtype=np.float64)
    if tg.size < 3:
        return float(np.interp(T, tg, v))
    lt = np.log(tg)
    x = float(np.log(max(T, 1e-6)))
    i = int(np.clip(np.searchsorted(lt, x) - 1, 1, tg.size - 2))
    return float(np.polyval(np.polyfit(lt[i-1:i+2], v[i-1:i+2], 2), x))


def rebuild(topk_logits, tail_logmass):
    """TRUE probabilities over the stored support -- deliberately NOT renormalised to 1.

    Renormalising changes the probability VALUES, which silently breaks every sampler that
    thresholds on an absolute probability (mirostat2's p >= 2^-mu, min_p's p >= max*v). On a
    test vector that shifted probabilities by up to 1.9e-2 and gave a 0.30 total-variation
    error. The returned vector sums to the mass the support actually holds.
    """
    z = topk_logits.astype(np.float64)
    kept = np.clip(1.0 - np.exp(np.minimum(tail_logmass, 0.0)), 1e-12, 1.0)
    m = z.max(-1, keepdims=True)
    e = np.exp(z - m)
    p = e / e.sum(-1, keepdims=True)
    return p * np.asarray(kept)[..., None], np.log(kept)


def entropy_from(p):
    return float(-(p * np.log(np.clip(p, 1e-30, None))).sum())


def temperature(md, H):
    if not md['dyn']:
        return md['t_hi']
    return md['t_lo'] + (md['t_hi'] - md['t_lo']) * (max(H, 0.0) / 11.93) ** md['t_exp']


def truncate(p, md, mu, zstd=None):
    tr, v = md['trunc'], md['tp']
    if tr == 'none':
        return np.ones_like(p, dtype=bool)
    if tr == 'min_p':
        return p >= p.max() * v
    if tr == 'typical':
        lp = np.log(np.clip(p, 1e-12, None))
        ent = -(p * lp).sum() if zstd is None else _TYPICAL_ENT[0]
        order = np.argsort(np.abs(-lp - ent))
        keep = (np.cumsum(p[order]) - p[order]) < v
        m = np.zeros_like(p, dtype=bool)
        m[order] = keep
        m[p.argmax()] = True
        return m
    if tr == 'top_nsigma':
        zz = np.log(np.clip(p, 1e-30, None))
        sd = zz.std() if zstd is None else zstd
        return zz >= zz.max() - v * sd
    return p >= 2.0 ** (-mu)


def xtc_mask(p, thr):
    B = p >= thr
    if int(B.sum()) < 2:
        return None
    idx = np.where(B)[0]
    keep = idx[np.argmin(p[idx])]
    m = np.ones_like(p, dtype=bool)
    m[idx] = False
    m[keep] = True
    return m


def induced(p_true, ids, md, st, order, hist_u, H=None, zstd=None, lse_grid=None,
            tgrid=None, logits=None, ent_grid=None):
    """The mode's induced distribution over the stored support, as TRUE probabilities.

    p_true sums to the mass the support holds, not to 1. Nothing here renormalises it before the
    sampler runs, because mirostat2 and typical sampling threshold on absolute probability and
    silently change behaviour if the scale moves.

    order='hf' applies temperature BEFORE truncation, so the sampler sees softmax(z/T) over the
    FULL vocabulary. That partition is not recoverable from the top-K, so simlab stores
    logsumexp(z/T) on a temperature grid and it is interpolated here in log space.
    """
    lg = np.log(np.clip(p_true, 1e-30, None))
    pen = {}
    if md['lam'] > 0 and hist_u:
        for tid, val in hist_u.items():
            w = np.where(ids == tid)[0]
            if w.size:
                d = md['lam'] * min(val, 3.0)
                lg[w[0]] -= d
                pen[int(w[0])] = d
    H = entropy_from(p_true) if H is None else H
    _TYPICAL_ENT[0] = H
    _ZSTD[0] = zstd
    T = temperature(md, H)

    kept = float(p_true.sum())

    def finish(b0, temp_after):
        msk = truncate(b0, md, st.get('mu', 5.0), _ZSTD[0])
        b = b0 * msk
        if b.sum() <= 0:
            b = b0.copy()
        # min_p and mirostat2 threshold on an ABSOLUTE probability. If that threshold falls
        # below the smallest probability in the stored support, tokens outside the support also
        # survive, and the generator's normaliser includes them while ours would not. Add the
        # tail mass back: exact when the threshold is below the whole tail, and the right
        # direction otherwise. Without this the surviving mass is understated and every
        # probability is inflated.
        tail_alive = 0.0
        if kept < 0.999:
            pmin = float(b0[b0 > 0].min())
            if md['trunc'] == 'typical':
                # typical sampling keeps cumulative mass just under tp BY CONSTRUCTION, so the
                # generator's surviving mass is about tp however far into the tail the typical
                # set reaches. Use that directly rather than trying to rebuild the set.
                tail_alive = max(0.0, float(md['tp']) - float(b.sum()))
            else:
                thr = None
                if md['trunc'] == 'min_p':
                    thr = float(b0.max() * md['tp'])
                elif md['trunc'] == 'mirostat2':
                    thr = 2.0 ** (-st.get('mu', 5.0))
                elif md['trunc'] == 'top_nsigma':
                    # zz >= zz.max() - v*sd is an absolute probability threshold in disguise
                    sd = _ZSTD[0] if _ZSTD[0] is not None else float(
                        np.log(np.clip(b0, 1e-30, None)).std())
                    thr = float(b0.max() * np.exp(-float(md['tp']) * sd))
                if thr is not None and thr <= pmin:
                    tail_alive = 1.0 - kept
        branches = [(1.0, b)]
        if md['xtc_q'] > 0:
            xm = xtc_mask(b / b.sum(), md['xtc_t'])
            if xm is not None and (b * xm).sum() > 0:
                branches = [(1 - md['xtc_q'], b), (md['xtc_q'], b * xm)]
        out = np.zeros_like(b0)
        for w, br in branches:
            q = br / max(br.sum() + tail_alive, 1e-30)
            if temp_after:
                q = np.clip(q, 1e-30, None) ** (1.0 / max(T, 1e-3))
                q = q / q.sum()
            out += w * q
        return out

    # trunc='none' keeps the ENTIRE vocabulary, so the surviving mass includes the tail and
    # renormalising over the 512-token support inflates every probability by 1/kept -- a pure
    # -log(kept) error, measured at 0.19-0.33 nats on real data because Qwen's top-512 holds
    # only 72-83% of the mass at these positions.
    #
    # With no truncation the policy is just a temperature transform, and for BOTH orders
    #     q_i = exp(z_i/T - logsumexp(z/T))
    # which the stored grid gives exactly. (For llama: p^(1/T) normalised over the full vocab
    # equals exp(z/T - lse_T) after the lse cancels.)
    if md['trunc'] == 'none' and lse_grid is not None and logits is not None:
        zt = logits.astype(np.float64)
        lse_T = grid_at(T, tgrid, lse_grid)
        if pen:
            idx = np.array(sorted(pen))
            dd = np.array([pen[i] for i in idx])
            old_m = np.exp(zt[idx] / max(T, 1e-3) - lse_T).sum()
            new_m = np.exp((zt[idx] - dd) / max(T, 1e-3) - lse_T).sum()
            lse_T += float(np.log(max(1.0 - old_m + new_m, 1e-12)))
            zt = zt.copy()
            zt[idx] -= dd
        q = np.exp(zt / max(T, 1e-3) - lse_T)
        if md['xtc_q'] > 0:
            xm = xtc_mask(q, md['xtc_t'])
            if xm is not None and (q * xm).sum() > 0:
                q = (1 - md['xtc_q']) * q + md['xtc_q'] * (q * xm) / (q * xm).sum() * q.sum()
        return q

    if order == 'llama':
        return finish(np.exp(lg), True)

    # hf: temperature first, over the full vocabulary
    if lse_grid is None or logits is None:
        sc = np.exp((lg - lg.max()) / max(T, 1e-3))
        return finish(sc / sc.sum(), False)
    # after tempering, the sampler's view of the distribution changes: typical sampling needs
    # the TEMPERED entropy and top_nsigma needs std(z)/T, not the untempered quantities
    if ent_grid is not None:
        _TYPICAL_ENT[0] = grid_at(T, tgrid, ent_grid)
    _ZSTD[0] = None if zstd is None else zstd / max(T, 1e-3)
    lse_T = grid_at(T, tgrid, lse_grid)
    zt = logits.astype(np.float64) / max(T, 1e-3)
    if pen:
        # the repetition penalty shifts the partition; correct it with the penalised tokens,
        # which are the only ones whose logits changed
        idx = np.array(sorted(pen))
        d = np.array([pen[i] for i in idx])
        old_m = np.exp(zt[idx] - lse_T).sum()
        new_m = np.exp((logits[idx].astype(np.float64) - d) / max(T, 1e-3) - lse_T).sum()
        lse_T = lse_T + float(np.log(max(1.0 - old_m + new_m, 1e-12)))
        zt[idx] = (logits[idx].astype(np.float64) - d) / max(T, 1e-3)
    return finish(np.exp(zt - lse_T), False)


def advance_state(st, md, pol_gamma, lp_base, tid):
    """Mirrors G.advance. Driven by the STORED base log-prob of the emitted token, not by a
    lookup in the support.

    This matters because mirostat2's mu evolves from the emitted token's surprisal every step.
    When the emitted token falls outside the stored top-512 -- which happens whenever the policy
    truncates loosely -- a support lookup finds nothing, the advance gets skipped, mu drifts
    away from the generator's, and every later step inherits the error. That compounding was
    worth 0.19-0.33 nats of median error on every mirostat task, while the stateless samplers
    were already correct to 1e-6.

    lp_base is the natural-log base-model probability of the emitted token, recorded at
    generation time, so it is available whether or not the token is in the support.
    """
    S = -float(lp_base) / np.log(2.0)
    if md['trunc'] == 'mirostat2':
        st['mu'] = float(np.clip(st.get('mu', 5.0) - md['eta'] * (S - md['tp']), 0.5, 20.0))
    if md['lam'] > 0:
        u = st.setdefault('u', {})
        g = pol_gamma
        for k in list(u):
            u[k] *= g
            if u[k] <= 1e-3:
                del u[k]
        u[int(tid)] = u.get(int(tid), 0.0) + g
    st['prev_surprisal'] = S * np.log(2)


def fresh_state(md):
    return dict(mu=2.0 * md['tp'] if md['trunc'] == 'mirostat2' else 5.0,
                u={}, prev_surprisal=0.0)


def induced_at(task, rec, p, ids, logits, md, st, tgrid):
    """induced() with every stored statistic threaded in. Use this, never induced() directly.

    Omitting any of H / zstd / lse_grid / ent_grid silently falls back to computing them from
    the truncated support, which is wrong by up to tens of nats.
    """
    return induced(p, ids, md, st, task['policy']['order'], st.get('u', {}),
                   H=rec['H'], zstd=rec['zstd'], lse_grid=rec.get('lse_grid'),
                   tgrid=tgrid, logits=logits, ent_grid=rec.get('ent_grid'))


def replay_trace(task, rec_steps, P, IDS, LG, tgrid, modes=None, fsm=None):
    """Walk one trace with a SINGLE evolving state, as the generator does.

    The generator creates one state per trace (G.init_state per prompt) and advances it every
    step with whichever mode is active, so mirostat's mu and the repetition counters carry
    across mode switches. Giving each mode its own state breaks that and is wrong.

    modes: per-mode parameter dicts to use (defaults to the task's true policy).
    fsm:   None to follow the recorded ground-truth modes; otherwise a policy dict whose guards
           and dwell/expiry drive the mode sequence forward.
    Returns (log-probs of the emitted tokens, modes used).
    """
    pol = task['policy']
    md_list = modes if modes is not None else pol['modes']
    st = fresh_state(md_list[0])
    out, used, cur, c = [], [], 0, 0
    for k, rec in enumerate(rec_steps):
        m = rec['mode'] if fsm is None else cur
        md = md_list[m]
        d = induced_at(task, rec, P[k], IDS[k], LG[k], md, st, tgrid)
        pos = np.where(IDS[k] == rec['tok'])[0]
        # the likelihood is only available for tokens inside the support, but the STATE must
        # advance either way or mirostat's mu desynchronises for the rest of the trace
        out.append(float(np.log(max(d[pos[0]], 1e-12))) if pos.size else float('nan'))
        advance_state(st, md, pol['gamma'], rec['lp_base'], rec['tok'])
        used.append(m)
        if fsm is not None:
            cur, c = step_fsm(fsm, cur, c, rec)
    return np.array(out), used


def step_fsm(pol, m, c, rec):
    """grammar_v3.fsm_step over a stored sensor record."""
    if c < pol['dwell'][m]:
        return m, c + 1
    if m != 0 and c >= pol['expiry'][m]:
        return 0, 0
    for g in pol['guards']:
        if g['src'] != m:
            continue
        v = rec[g['sensor']]
        if (v >= g['thr']) if g['cmp'] == 'ge' else (v < g['thr']):
            return g['dst'], 0
    return m, c + 1


def verify_against_generator(task, npz, tgrid, ntrace=2, n=256):
    """Recompute stored per-step log-probs from the stored distribution and compare.

    This is the audit that the CPU path reproduces the GPU generator. It must pass before any
    fit is trusted: a large gap means the stored statistics are being used wrongly, or the
    top-K truncation is material, and every downstream number inherits it.
    """
    errs = []
    for ti, tr in enumerate(task['traces'][:ntrace]):
        steps = tr['steps'][:n]
        Pm, _ = rebuild(npz['cal_lg'][ti][:len(steps)], npz['cal_tail'][ti][:len(steps)])
        lp, _ = replay_trace(task, steps, Pm, npz['cal_ids'][ti], npz['cal_lg'][ti], tgrid)
        for k, step in enumerate(steps):
            if np.isfinite(lp[k]):
                errs.append(abs(lp[k] - step['lp_policy']))
    return np.array(errs) if errs else np.array([0.0])

In [ ]:
%%writefile /kaggle/working/simlab.py
"""Fresh simulation recording enough that no second GPU run is ever needed.

    python simlab.py

WHY A FRESH RUN. The released dataset is unreproducible: make_dataset.py samples with
torch.multinomial and never calls torch.manual_seed, so the hidden policies existed only in that
process's memory and no mode labels can be recovered for it.

THE DESIGN PRINCIPLE. Store the base model's next-token distribution at every step. Every sampler
in the grammar -- min_p, typical, top_nsigma, mirostat2, XTC, repetition penalty, dynamic
temperature -- is a deterministic function of that distribution and its parameters. With it
stored, any policy's induced distribution at any step is recomputable offline on CPU, so every
fit, baseline, ablation and diagnostic becomes a CPU job. It is the only quantity here that
cannot be reconstructed without repeating the GPU run.

WHAT IS RECORDED, per step of every calibration trace and every scored prefix:
  topk_ids / topk_logits / tail_logmass   the base distribution, truncated to TOPK
  sensors                                 H, G, U, D, P -- what the FSM gates on
  mode, dwell, mu                         ground truth state
  margins, predicates, fired, blocked     transition provenance: why the FSM did what it did
  logp_under_mode[M]                      the emitted token's log-prob under EVERY mode
  tv_between_modes                        pairwise total variation between mode distributions
  u                                       the uniform draw the sampler consumed

logp_under_mode and tv_between_modes are the per-step mode discriminability. From them the
Bayes-optimal mode posterior under the TRUE parameters is computable, which separates "the
observation does not determine the mode" from "the predictor is bad" -- the decomposition that
makes a negative result attributable rather than merely reported.

CONDITIONS, balanced across tasks:
  modes      1 (genuinely stateless) / 2 / 3
  dwell      on (dwell locks + expiry, non-Markov in the sensors) / off (mode is a function of
             the current sensor vector, so pointwise identifiable)
  margin     thresholds placed at designed quantiles of each sensor's MEASURED distribution:
             well-bracketed (40-60th pct, many crossings) or poorly-bracketed (5th/95th, few).
             Two-pass: a base-model-only trace measures the sensor distribution first. This
             converts an incidental observation -- that thresholds are identifiable only to an
             interval -- into a controlled main effect, and cannot be added afterwards.

Every task seeds torch from its own index, and the consumed uniform draws are stored, so any
trace can be verified without regenerating it.
"""
import os, sys, csv, json, math, random, time, hashlib, platform
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import grammar_v3 as G
import probe as M

OUT = os.environ.get('OUT', './sim')
NTASK = int(os.environ.get('NTASK', 250))
NTRACE = int(os.environ.get('NTRACE', 16))
TRACELEN = int(os.environ.get('TRACELEN', 256))
NITEM = int(os.environ.get('NITEM', 40))
OVERSAMPLE = float(os.environ.get('OVERSAMPLE', 2.5))
PREFIX_LEN = int(os.environ.get('PREFIX_LEN', 24))
TOPK = int(os.environ.get('TOPK', 512))
NPROMPT_POOL = int(os.environ.get('NPROMPT_POOL', 48))   # prompts crossed with policies
NREPLICATE = int(os.environ.get('NREPLICATE', 10))       # duplicate policies, new trace seeds
VERIFY_FULL = int(os.environ.get('VERIFY_FULL', 5))      # tasks also checked against full vocab
SEED = 71294
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
T0 = time.time()
TGRID_T = None   # set in main(), after the device is known
SENSORS = ['H', 'G', 'U', 'D', 'P']
# temperature grid for the stored partition; bounds cover BOUNDS['t_lo']..['t_hi'] in grammar_v3
TGRID = [0.25 * (1.1 ** k) for k in range(26)]


def log(m):
    print('[%7.1fs] %s' % (time.time() - T0, m), flush=True)


# ------------------------------------------------------------------ conditions

def condition_of(i):
    """Balanced assignment. dwell is vacuous at M=1, so the cell structure is 5-fold."""
    cells = [(1, False), (2, False), (2, True), (3, False), (3, True)]
    m, dw = cells[i % len(cells)]
    margin = 'well' if (i // len(cells)) % 2 == 0 else 'poor'
    return dict(M=m, dwell=dw, margin=margin)


def canonical(pol):
    """Mode-relabel-invariant form, so policies identical up to relabelling hash identically.

    Without this, two equivalent policies look distinct and apparent diversity is inflated --
    which matters because the central claim is cross-task amortization and a reviewer will ask
    for score against distance-to-nearest-training-policy.
    """
    order = sorted(range(pol['M']), key=lambda i: (pol['modes'][i]['trunc'],
                                                   round(pol['modes'][i]['tp'], 4)))
    rank = {o: j for j, o in enumerate(order)}
    md = [{k: v for k, v in pol['modes'][o].items()} for o in order]
    gd = sorted([dict(src=rank[g['src']], dst=rank[g['dst']], sensor=g['sensor'],
                      cmp=g['cmp'], thr=round(g['thr'], 4)) for g in pol['guards']],
                key=lambda g: (g['src'], g['dst'], g['sensor']))
    c = dict(M=pol['M'], order=pol['order'], gamma=pol['gamma'], modes=md, guards=gd,
             dwell=[pol['dwell'][o] for o in order], expiry=[pol['expiry'][o] for o in order])
    return c, hashlib.sha1(json.dumps(c, sort_keys=True).encode()).hexdigest()[:16]


def build_policy(rng, cond, sensor_quantiles):
    """Draw a policy, then force the condition and place thresholds at designed quantiles."""
    p = G.sample_policy(rng)
    p['M'] = cond['M']
    while len(p['modes']) < p['M']:
        p['modes'].append(G.sample_mode(rng))
    p['modes'] = p['modes'][:p['M']]
    p['dwell'] = ([rng.choice([0, 3]) for _ in range(p['M'])] if cond['dwell']
                  else [0] * p['M'])
    p['expiry'] = ([rng.choice([8, 16, 32]) for _ in range(p['M'])] if cond['dwell']
                   else [999] * p['M'])
    if p['M'] == 1:
        p['guards'] = []
        return p
    pairs = [(0, 1), (1, 0)] if p['M'] == 2 else [(0, 1), (1, 2), (1, 0), (2, 0)]
    guards = []
    for src, dst in pairs:
        s = rng.choice(SENSORS)
        q = sensor_quantiles[s]
        if cond['margin'] == 'well':
            lo, hi = np.percentile(q, 40), np.percentile(q, 60)
            thr = float(rng.uniform(lo, hi))
        else:
            thr = float(np.percentile(q, 5 if rng.random() < 0.5 else 95))
        guards.append(dict(src=src, dst=dst, sensor=s, cmp=rng.choice(['ge', 'lt']),
                           thr=round(thr, 4)))
    p['guards'] = guards
    return p


# ------------------------------------------------------------------ rollout

def predicate_state(pol, s, st):
    """Which gates are true, by how much, and what the FSM is allowed to do about it."""
    vals, marg = [], []
    for g in pol['guards']:
        v = s[g['sensor']]
        true = (v >= g['thr']) if g['cmp'] == 'ge' else (v < g['thr'])
        d = v - g['thr']
        vals.append(bool(true))
        marg.append(round(float(d if g['cmp'] == 'ge' else -d), 5))
    blocked = bool(st['c'] < pol['dwell'][st['m']] and any(
        vals[j] for j, g in enumerate(pol['guards']) if g['src'] == st['m']))
    expiring = bool(st['m'] != 0 and st['c'] >= pol['expiry'][st['m']])
    return vals, marg, blocked, expiring


@torch.no_grad()
def rollout(model, tok, prompts, pol, nstep, seed, want_logits=True):
    """Generate nstep tokens per prompt, recording everything.

    SHADOW STATE CONVENTION: a separate sampler state is evolved for every mode as if that mode
    were always active, so logp_under_mode is the log-prob that mode's sampler would have given
    had it been running throughout. The alternative -- freezing inactive state -- makes the
    counterfactual depend on occupancy history and is not what a mode-posterior needs. Declared
    here and stored in the task record.
    """
    g = torch.Generator(device=DEV).manual_seed(seed)
    n = len(prompts)
    enc = tok([t for _, t in prompts], return_tensors='pt', padding=True).to(DEV)
    ids, attn = enc.input_ids, enc.attention_mask
    pos = (attn.cumsum(-1) - 1).clamp_min(0)
    past, cur = None, ids
    st = [G.init_state(pol) for _ in range(n)]
    shadow = [[G.init_state(pol) for _ in range(pol['M'])] for _ in range(n)]
    toks = [[] for _ in range(n)]
    rec = [[] for _ in range(n)]
    # nstep + 1: the last slot is the SCORED position, reached by the extra forward below.
    # Truncation samplers need the whole distribution to decide what survives, so storing only
    # the candidates' base log-probs there would make the scored step uncomputable offline.
    tk_ids = np.zeros((n, nstep + 1, TOPK), dtype=np.int32)
    tk_lg = np.zeros((n, nstep + 1, TOPK), dtype=np.float16)
    tail = np.zeros((n, nstep + 1), dtype=np.float32)
    ent_last = np.zeros((n, len(TGRID)), dtype=np.float32)
    lse_last = np.zeros((n, len(TGRID)), dtype=np.float32)
    stat_last = np.zeros((n, 2), dtype=np.float32)      # H, zstd at the scored position

    for step in range(nstep):
        out = model(input_ids=cur, attention_mask=attn, position_ids=pos,
                    past_key_values=past, use_cache=True)
        past = out.past_key_values
        z = out.logits[:, -1, :].half()
        nxt = []
        for b in range(n):
            zf = z[b].float()
            p_raw = torch.softmax(zf, -1)
            lse = torch.logsumexp(zf, -1)
            tv, ti = torch.topk(zf, TOPK)
            # top_nsigma thresholds on the std of log-probs over the FULL vocabulary and
            # typical sampling on the FULL entropy. Neither survives truncation to top-K -- on a
            # test vector the std was 3.00 full vs 0.76 truncated -- so both are stored.
            zstd = float(torch.log(p_raw.clamp_min(1e-30)).std())
            # 'hf'-order policies apply temperature BEFORE truncation, so the sampler sees
            # softmax(z/T) over the FULL vocabulary. That partition cannot be recovered from the
            # top-K, so it is stored on a temperature grid and interpolated in log space.
            zg = zf[None, :] / TGRID_T[:, None]
            lseg = torch.logsumexp(zg, -1)
            # typical sampling thresholds on the entropy of the distribution it is given, which
            # for hf order is the TEMPERED one. Stored on the same grid as the partition.
            pg = torch.softmax(zg, -1)
            entg = -(pg * torch.log(pg.clamp_min(1e-30))).sum(-1)
            if want_logits:
                tk_ids[b, step] = ti.cpu().numpy()
                tk_lg[b, step] = tv.cpu().numpy().astype(np.float16)
                kept = torch.exp(tv - lse).sum()
                tail[b, step] = float(torch.log((1.0 - kept).clamp_min(1e-30)))

            s = G.sensors(p_raw, z[b], st[b], step)
            vals, marg, blocked, expiring = predicate_state(pol, s, st[b])
            m_before = st[b]['m']
            d, praw, md = G.policy_dist(z[b], pol, st[b], step)
            u = float(torch.rand(1, generator=g, device=DEV))
            t = int(torch.searchsorted(torch.cumsum(d, 0), torch.tensor(u, device=DEV)).clamp(
                0, d.numel() - 1))

            # counterfactual: the emitted token's log-prob under every mode, and how far apart
            # the modes' induced distributions are at this step
            lpm, dists = [], []
            # the counterfactual policy must have no guards, no dwell and no expiry, or the FSM
            # would reset a pinned shadow mode to 0 and the counterfactual would be wrong
            pinned = {**pol, 'guards': [], 'dwell': [0] * pol['M'],
                      'expiry': [10 ** 9] * pol['M']}
            for mi in range(pol['M']):
                stt = shadow[b][mi]
                stt['m'], stt['c'] = mi, 0
                dm, prm, mdm = G.policy_dist(z[b], pinned, stt, step)
                lpm.append(round(float(torch.log(dm[t].clamp_min(1e-12))), 5))
                dists.append(dm)
            tvs = []
            for a in range(pol['M']):
                for c in range(a + 1, pol['M']):
                    tvs.append(round(float(0.5 * (dists[a] - dists[c]).abs().sum()), 5))

            rec[b].append(dict(H=round(s['H'], 5), G=round(s['G'], 5), U=round(s['U'], 5),
                               zstd=round(zstd, 5), lse_grid=[round(float(x), 5) for x in lseg],
                               ent_grid=[round(float(x), 5) for x in entg], kept=round(float(torch.exp(tv - lse).sum()), 6),
                               D=s['D'], P=s['P'], mode=m_before, mode_after=st[b]['m'],
                               dwell=st[b]['c'], mu=round(st[b]['mu'], 4),
                               pred=vals, margins=marg, blocked=blocked, expiring=expiring,
                               u=round(u, 7), tok=t,
                               lp_policy=round(float(torch.log(d[t].clamp_min(1e-12))), 5),
                               lp_base=round(float(torch.log(p_raw[t].clamp_min(1e-12))), 5),
                               lp_mode=lpm, tv=tvs))
            G.advance(st[b], pol, md, praw, t)
            for mi in range(pol['M']):
                G.advance(shadow[b][mi], pinned, pol['modes'][mi], praw, t)
            toks[b].append(t)
            nxt.append(t)
        cur = torch.tensor(nxt, device=DEV).unsqueeze(1)
        attn = torch.cat([attn, torch.ones(n, 1, dtype=attn.dtype, device=DEV)], 1)
        pos = pos[:, -1:] + 1

    out = model(input_ids=cur, attention_mask=attn, position_ids=pos,
                past_key_values=past, use_cache=True)
    zl = out.logits[:, -1, :].half()
    if want_logits:
        for b in range(n):
            zf = zl[b].float()
            pr = torch.softmax(zf, -1)
            lse = torch.logsumexp(zf, -1)
            tv, ti = torch.topk(zf, TOPK)
            tk_ids[b, nstep] = ti.cpu().numpy()
            tk_lg[b, nstep] = tv.cpu().numpy().astype(np.float16)
            tail[b, nstep] = float(torch.log((1.0 - torch.exp(tv - lse).sum()).clamp_min(1e-30)))
            zg = zf[None, :] / TGRID_T[:, None]
            lse_last[b] = torch.logsumexp(zg, -1).cpu().numpy()
            pg = torch.softmax(zg, -1)
            ent_last[b] = (-(pg * torch.log(pg.clamp_min(1e-30))).sum(-1)).cpu().numpy()
            stat_last[b] = [float(-(pr * torch.log(pr.clamp_min(1e-30))).sum()),
                            float(torch.log(pr.clamp_min(1e-30)).std())]
    return toks, rec, st, zl, (tk_ids, tk_lg, tail, lse_last, ent_last, stat_last)


@torch.no_grad()
def measure_sensors(model, tok, prompts, nstep, seed):
    """Pass 1: base-model-only generation, to measure each sensor's realised distribution.

    Doubles as the per-prompt reference trace needed to show the sensor trajectory carries
    policy information rather than prompt information.
    """
    flat = G.sample_policy(random.Random(0))
    flat['M'] = 1
    flat['guards'] = []
    flat['dwell'] = [0]
    flat['expiry'] = [999]
    flat['modes'] = [{**flat['modes'][0], 'trunc': 'none', 'dyn': False, 't_hi': 1.0,
                      'lam': 0.0, 'xtc_q': 0.0}]
    toks, rec, _, _, _ = rollout(model, tok, prompts, flat, nstep, seed, want_logits=False)
    q = {s: np.array([r[s] for tr in rec for r in tr], dtype=np.float32) for s in SENSORS}
    return q, toks, rec


def main():
    os.makedirs(OUT, exist_ok=True)
    cal_prompts = M.load_prompts(os.environ.get('PC', './raw/prompts_calibration.csv'))
    qry_prompts = M.load_prompts(os.environ.get('PQ', './raw/prompts_query.csv'))
    rngp = random.Random(SEED)
    pool_c = [cal_prompts[rngp.randrange(len(cal_prompts))] for _ in range(NPROMPT_POOL)]
    pool_q = [qry_prompts[rngp.randrange(len(qry_prompts))] for _ in range(NPROMPT_POOL)]
    log('prompt pool: %d calibration, %d query (crossed with policies)' % (len(pool_c), len(pool_q)))

    name = os.environ.get('MODEL', 'Qwen/Qwen2.5-0.5B')
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'left'
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if DEV == 'cuda' else torch.float32).to(DEV).eval()
    global TGRID_T
    TGRID_T = torch.tensor(TGRID, device=DEV, dtype=torch.float32)

    env = dict(model=name, device=DEV, torch=torch.__version__, python=platform.python_version(),
               dtype=str(next(model.parameters()).dtype), topk=TOPK,
               shadow_state='evolved as if always active', seed=SEED, tgrid=TGRID,
               tracelen=TRACELEN, ntrace=NTRACE, nitem=NITEM, prefix_len=PREFIX_LEN)
    json.dump(env, open(os.path.join(OUT, 'env.json'), 'w'), indent=1)
    log('env: %s' % json.dumps(env))
    log('budgets derivable by subsetting: 8x32=256, 8x128=1024, 16x128=2048, 16x256=%d tokens'
        % (NTRACE * TRACELEN))

    path = os.path.join(OUT, 'tasks.jsonl')
    done = set()
    if os.path.exists(path):
        done = {json.loads(l)['task_id'] for l in open(path, encoding='utf-8')}
        log('resuming: %d tasks present' % len(done))
    f = open(path, 'a', encoding='utf-8')
    kept, skipped = 0, 0
    seen_policies = {}

    for i in range(NTASK):
        tid = 'T%05d' % i
        if tid in done:
            continue
        cond = condition_of(i)
        rng = random.Random(SEED * 7919 + i)
        # crossing: every task draws its prompts from the shared pool at a rotating offset, so
        # each prompt recurs across policies and each policy sees many prompts
        off = (i * 7) % NPROMPT_POOL
        cp = [pool_c[(off + k) % NPROMPT_POOL] for k in range(NTRACE)]
        qp = [pool_q[(off + k) % NPROMPT_POOL] for k in range(int(NITEM * OVERSAMPLE))]

        # replicates: reuse an earlier task's policy with fresh trace seeds
        rep_of = None
        if i >= NTASK - NREPLICATE and seen_policies:
            rep_of = sorted(seen_policies)[(i - (NTASK - NREPLICATE)) % len(seen_policies)]
            pol = seen_policies[rep_of]
            cond = dict(M=pol['M'], dwell=any(d > 0 for d in pol['dwell']),
                        margin='inherited', replicate_of=rep_of)
        else:
            q, btoks, brec = measure_sensors(model, tok, cp[:4], 64, SEED * 29 + i)
            pol = None
            for attempt in range(60):
                cand = build_policy(random.Random(SEED * 1000003 + i * 997 + attempt), cond, q)
                if cand['M'] == 1:
                    pol = cand
                    break
                _, rc, _, _, _ = rollout(model, tok, cp[:4], cand, 32,
                                         SEED * 13 + i * 101 + attempt, want_logits=False)
                modes = [r['mode'] for tr in rc for r in tr]
                sh = [modes.count(m) / len(modes) for m in range(cand['M'])]
                if min(sh) >= 0.05:
                    pol = cand
                    break
            if pol is None:
                skipped += 1
                log('  %s SKIPPED: no policy reached 5%% occupancy in all modes' % tid)
                continue

        ctoks, crec, _, _, cl = rollout(model, tok, cp, pol, TRACELEN, SEED * 17 + i * 7919)
        ptoks, prec, pst, pz, pl = rollout(model, tok, qp, pol, PREFIX_LEN,
                                           SEED * 19 + i * 104729)
        items = []
        for b in range(len(qp)):
            got = M.scored_item(pz[b], pol, pst[b], PREFIX_LEN, rng)
            if got is None:
                continue
            cands, ti, pt, pb, maxp, _ = got
            if maxp < M.MIN_MAXP:
                continue
            lp = torch.log_softmax(pz[b].float(), -1)
            rk = torch.argsort(torch.argsort(-pz[b].float()))
            items.append(dict(idx=b, prompt_id=qp[b][0], prefix=ptoks[b], candidates=cands,
                              true_index=ti, lp_policy=round(math.log(max(pt, 1e-12)), 6),
                              lp_base=round(math.log(max(pb, 1e-12)), 6),
                              cand_lp_base=[round(float(lp[c]), 5) for c in cands],
                              cand_rank=[int(rk[c]) for c in cands],
                              true_is_argmax=bool(cands[ti] == int(pz[b].argmax())),
                              steps=prec[b], mode_at_item=pst[b]['m'],
                              scored=dict(H=round(float(pl[5][b][0]), 5),
                                          zstd=round(float(pl[5][b][1]), 5),
                                          lse_grid=[round(float(x), 5) for x in pl[3][b]],
                                          ent_grid=[round(float(x), 5) for x in pl[4][b]])))
            if len(items) >= NITEM:
                break
        if not items:
            skipped += 1
            log('  %s SKIPPED: no items survived the MIN_MAXP filter' % tid)
            continue

        canon, chash = canonical(pol)
        st_all = [s for tr in crec for s in tr]
        occ = [float(np.mean([s['mode'] == m for s in st_all])) for m in range(pol['M'])]
        switches = int(np.sum([tr[k]['mode'] != tr[k - 1]['mode']
                               for tr in crec for k in range(1, len(tr))]))
        rec_out = dict(task_id=tid, condition=cond, policy=pol, canonical=canon,
                       policy_hash=chash, occupancy=occ, switches=switches,
                       degenerate=bool(min(occ) < 0.05),
                       trace_prompts=[p[0] for p in cp],
                       traces=[dict(prompt_id=cp[b][0], tokens=ctoks[b], steps=crec[b])
                               for b in range(len(cp))],
                       items=items)
        f.write(json.dumps(rec_out) + '\n')
        f.flush()
        np.savez_compressed(os.path.join(OUT, '%s.npz' % tid),
                            cal_ids=cl[0], cal_lg=cl[1], cal_tail=cl[2],
                            itm_ids=pl[0][[it['idx'] for it in items]],
                            itm_lg=pl[1][[it['idx'] for it in items]],
                            itm_tail=pl[2][[it['idx'] for it in items]])
        if rep_of is None:
            seen_policies[tid] = pol
        kept += 1
        if kept % 10 == 0 or kept == 1:
            log('  %s | %d/%d | M=%d dwell=%s margin=%s | occ=%s switches=%d | %d items%s'
                % (tid, kept, NTASK, pol['M'], cond['dwell'], cond['margin'],
                   ','.join('%.2f' % x for x in occ), switches, len(items),
                   ' | REPLICATE of %s' % rep_of if rep_of else ''))
    f.close()
    log('done: %d kept, %d skipped' % (kept, skipped))
    summarise(path)


def summarise(path):
    rows = [json.loads(l) for l in open(path, encoding='utf-8')]
    if not rows:
        return
    log('')
    log('==== RUN SUMMARY ====')
    cells = {}
    for r in rows:
        k = (r['condition']['M'], r['condition']['dwell'], r['condition']['margin'])
        cells[k] = cells.get(k, 0) + 1
    for k in sorted(cells):
        log('  M=%d dwell=%-5s margin=%-4s : %d tasks' % (k[0], k[1], k[2], cells[k]))
    deg = sum(r['degenerate'] for r in rows)
    log('  degenerate (a mode below 5%% occupancy): %d of %d -- PRE-REGISTERED EXCLUSION'
        % (deg, len(rows)))
    hashes = {}
    for r in rows:
        hashes[r['policy_hash']] = hashes.get(r['policy_hash'], 0) + 1
    log('  distinct canonical policies: %d (duplicates: %d)'
        % (len(hashes), sum(v - 1 for v in hashes.values())))
    marg = np.array([abs(x) for r in rows for tr in r['traces'] for s in tr['steps']
                     for x in s['margins']])
    if marg.size:
        log('  guard margin |sensor-threshold|: median %.3f | frac < 0.05: %.4f'
            % (float(np.median(marg)), float((marg < 0.05).mean())))
    tv = np.array([x for r in rows for tr in r['traces'] for s in tr['steps'] for x in s['tv']])
    if tv.size:
        log('  inter-mode TV per step: median %.4f | frac < 0.01 (informationless): %.4f'
            % (float(np.median(tv)), float((tv < 0.01).mean())))
        log('    a high informationless fraction means the modes are not distinguishable at')
        log('    those steps by ANY method, which bounds what mode inference can achieve.')
    sw = np.array([r['switches'] for r in rows])
    log('  mode switches per task: median %d, 10-90%% %d-%d'
        % (int(np.median(sw)), int(np.percentile(sw, 10)), int(np.percentile(sw, 90))))
    log('  items per task: mean %.1f' % float(np.mean([len(r['items']) for r in rows])))


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /kaggle/working/patch_scored.py
"""Backfill the scored position into a simulation generated before simlab stored it.

    python patch_scored.py

The first version of simlab stored the base distribution for the 24 prefix steps but not for the
scored position itself -- the extra forward pass. Truncation samplers need the whole distribution
to decide what survives, so without it the induced distribution at exactly the position being
predicted cannot be computed offline.

Regenerating costs ~4 GPU hours. This costs one short forward pass per item (prompt + 24 tokens),
so a few minutes, because the prefix tokens are already recorded and the model is deterministic
given them.

Idempotent: tasks that already carry a 'scored' record are skipped.
"""
import os, json, math, time, shutil
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import probe as M

SIM = os.environ.get('SIM', './sim')
BATCH = int(os.environ.get('BATCH', 16))
T0 = time.time()
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'


def log(m):
    print('[%6.1fs] %s' % (time.time() - T0, m), flush=True)


@torch.no_grad()
def main():
    env = json.load(open(os.path.join(SIM, 'env.json')))
    TOPK, TGRID = env['topk'], env['tgrid']
    tg = torch.tensor(TGRID, device=DEV, dtype=torch.float32)
    prompts = {}
    for f in (os.environ.get('PC', './raw/prompts_calibration.csv'),
              os.environ.get('PQ', './raw/prompts_query.csv')):
        for pid, txt in M.load_prompts(f):
            prompts[pid] = txt

    tok = AutoTokenizer.from_pretrained(env['model'])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'left'
    model = AutoModelForCausalLM.from_pretrained(
        env['model'], dtype=torch.float16 if DEV == 'cuda' else torch.float32).to(DEV).eval()
    log('model loaded; TOPK=%d, %d grid points' % (TOPK, len(TGRID)))

    src = os.path.join(SIM, 'tasks.jsonl')
    rows = [json.loads(l) for l in open(src, encoding='utf-8')]
    log('%d tasks on disk' % len(rows))
    todo = [r for r in rows if r['items'] and 'scored' not in r['items'][0]]
    log('%d need backfilling' % len(todo))

    for n, r in enumerate(todo):
        items = r['items']
        seqs = [tok(prompts[it['prompt_id']]).input_ids + list(it['prefix']) for it in items]
        npz = dict(np.load(os.path.join(SIM, '%s.npz' % r['task_id'])))
        T = npz['itm_lg'].shape[1]
        if T == len(items[0]['steps']):              # no slot for the scored step yet
            npz['itm_ids'] = np.concatenate(
                [npz['itm_ids'], np.zeros((len(items), 1, TOPK), np.int32)], 1)
            npz['itm_lg'] = np.concatenate(
                [npz['itm_lg'], np.zeros((len(items), 1, TOPK), np.float16)], 1)
            npz['itm_tail'] = np.concatenate(
                [npz['itm_tail'], np.zeros((len(items), 1), np.float32)], 1)
        last = npz['itm_lg'].shape[1] - 1

        for b0 in range(0, len(seqs), BATCH):
            chunk = seqs[b0:b0 + BATCH]
            w = max(len(x) for x in chunk)
            ids = torch.full((len(chunk), w), tok.pad_token_id, dtype=torch.long)
            att = torch.zeros((len(chunk), w), dtype=torch.long)
            for j, x in enumerate(chunk):            # left padding, matching the generator
                ids[j, w - len(x):] = torch.tensor(x)
                att[j, w - len(x):] = 1
            ids, att = ids.to(DEV), att.to(DEV)
            posi = (att.cumsum(-1) - 1).clamp_min(0)
            z = model(input_ids=ids, attention_mask=att,
                      position_ids=posi).logits[:, -1, :].float()
            for j in range(len(chunk)):
                zf = z[j]
                pr = torch.softmax(zf, -1)
                lse = torch.logsumexp(zf, -1)
                tv, ti = torch.topk(zf, TOPK)
                npz['itm_ids'][b0 + j, last] = ti.cpu().numpy()
                npz['itm_lg'][b0 + j, last] = tv.cpu().numpy().astype(np.float16)
                npz['itm_tail'][b0 + j, last] = float(
                    torch.log((1.0 - torch.exp(tv - lse).sum()).clamp_min(1e-30)))
                zg = zf[None, :] / tg[:, None]
                pg = torch.softmax(zg, -1)
                items[b0 + j]['scored'] = dict(
                    H=round(float(-(pr * torch.log(pr.clamp_min(1e-30))).sum()), 5),
                    zstd=round(float(torch.log(pr.clamp_min(1e-30)).std()), 5),
                    lse_grid=[round(float(x), 5) for x in torch.logsumexp(zg, -1)],
                    ent_grid=[round(float(x), 5) for x in
                              -(pg * torch.log(pg.clamp_min(1e-30))).sum(-1)])
        np.savez_compressed(os.path.join(SIM, '%s.npz' % r['task_id']), **npz)
        if (n + 1) % 10 == 0 or n == 0:
            log('  %d/%d tasks backfilled' % (n + 1, len(todo)))

    shutil.copy(src, src + '.bak')
    with open(src, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r) + '\n')
    log('rewrote %s (backup at %s.bak)' % (src, src))

    # audit: the candidate base log-probs were recorded at generation time, so they are an
    # independent check that this re-run reproduced the same forward pass
    errs = []
    for r in rows[:20]:
        if not r['items'] or 'scored' not in r['items'][0]:
            continue
        npz = np.load(os.path.join(SIM, '%s.npz' % r['task_id']))
        last = npz['itm_lg'].shape[1] - 1
        for n, it in enumerate(r['items']):
            lg = npz['itm_lg'][n][last].astype(np.float64)
            ids = npz['itm_ids'][n][last]
            lse = np.log(np.exp(lg - lg.max()).sum()) + lg.max()
            kept = 1.0 - np.exp(min(npz['itm_tail'][n][last], 0.0))
            for c, want in zip(it['candidates'], it['cand_lp_base']):
                w = np.where(ids == c)[0]
                if w.size:
                    errs.append(abs((lg[w[0]] - lse + np.log(max(kept, 1e-12))) - want))
    e = np.array(errs) if errs else np.array([0.0])
    log('')
    log('AUDIT vs the cand_lp_base recorded during generation:')
    log('  median %.2e | 99th %.2e | max %.2e nats over %d candidates'
        % (float(np.median(e)), float(np.percentile(e, 99)), float(e.max()), e.size))
    log('  under ~1e-2 confirms the re-run reproduced the original forward pass')


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /kaggle/working/gate_c1a.py
"""C1a: the mode-oracle control. Run this BEFORE training any predictor.

    python gate_c1a.py

C1a reveals the TRUE mode label at every step of the calibration traces -- and nowhere else --
then fits the policy with that supervision and predicts the scored items. It is exactly what a
perfect mode predictor would hand downstream, so it is the real ceiling of the amortized
approach, not the true-policy oracle at 1.0.

WHY IT IS A GATE. The kill condition for the amortized experiment is 0.6 of headroom, expressed
against the true-policy oracle. If C1a itself lands below about 0.7, then 0.6 is unreachable by
construction, the experiment adjudicates nothing, and training a predictor adds no information.
The finding would instead be: even with the modes revealed, passive single-sample observation
does not identify these policies.

Pre-committed, before any number was seen:

    PASS  C1a >= 0.70 of headroom  -> build the predictor; it must then reach >= 0.60 of
                                      headroom AND >= 0.70 x C1a
    FAIL  C1a <  0.70              -> stop. Report the identifiability result instead.

All CPU: simlab stored the base distribution at every step.
"""
import os, json, math, time, random
import numpy as np

import policy_cpu as P

SIM = os.environ.get('SIM', './sim')
NTASK = int(os.environ.get('GATE_NTASK', 40))
FITSTEPS = int(os.environ.get('FITSTEPS', 1024))   # cap on calibration steps per fit
PROPOSALS = int(os.environ.get('PROPOSALS', 120))
BUDGET = os.environ.get('BUDGET', '16x256')
EPS = float(os.environ.get('EPS', 0.03))
THR_C1A = 0.70
# Tasks whose induced distribution cannot be reconstructed from the stored top-512 are excluded.
# This is not a convenience filter: mirostat2 thresholds on p >= 2^-mu, which at mu ~ 18 is
# 3.8e-6 and sits far below rank 512, so its keep-set extends into the tail the record does not
# hold -- measured at up to 23% of emitted tokens falling outside the support. Every other
# sampler reconstructs to <= 1.9e-3 nats. The criterion is the MEASURED error, not the sampler
# name, because a mirostat task whose mu stays high is perfectly reconstructible and is kept.
AUDIT_MAX = float(os.environ.get('AUDIT_MAX', 5e-3))
SEED = 71294
T0 = time.time()
KEYS = ('tp', 'eta', 't_lo', 't_hi', 't_exp', 'xtc_q', 'xtc_t', 'lam')


def log(m):
    print('[%7.1fs] %s' % (time.time() - T0, m), flush=True)


def budget_traces(task, spec, cap):
    """Whole traces, truncated. NOT a random sample of steps.

    The generator creates one state per trace and evolves it every step, so mirostat's mu and
    the repetition counters depend on the whole preceding trace. Subsampling steps breaks that
    continuity and makes every likelihood wrong.
    """
    n, ln = (int(x) for x in spec.split('x'))
    out, used = [], 0
    for ti in range(min(n, len(task['traces']))):
        steps = task['traces'][ti]['steps'][:ln]
        if used + len(steps) > cap and out:
            break
        out.append((ti, steps))
        used += len(steps)
    return out


def load(limit, tgrid):
    """Usable tasks only: non-degenerate, with items, and reconstructible on the CPU path."""
    rows, drop = [], dict(degenerate=0, no_items=0, audit=0)
    dropped_truncs = {}
    for line in open(os.path.join(SIM, 'tasks.jsonl'), encoding='utf-8'):
        r = json.loads(line)
        if r.get('degenerate'):
            drop['degenerate'] += 1
            continue
        if not r['items']:
            drop['no_items'] += 1
            continue
        if 'scored' not in r['items'][0]:
            raise SystemExit('tasks lack the scored-position record; run patch_scored.py first')
        npz = np.load(os.path.join(SIM, '%s.npz' % r['task_id']))
        e = float(np.median(P.verify_against_generator(r, npz, tgrid, ntrace=1, n=128)))
        if e > AUDIT_MAX:
            drop['audit'] += 1
            for m in r['policy']['modes']:
                dropped_truncs[m['trunc']] = dropped_truncs.get(m['trunc'], 0) + 1
            continue
        r['_audit'] = e
        rows.append(r)
        if len(rows) >= limit:
            break
    log('excluded: %d degenerate, %d itemless, %d above audit %.0e'
        % (drop['degenerate'], drop['no_items'], drop['audit'], AUDIT_MAX))
    if dropped_truncs:
        log('  truncations among excluded tasks: %s'
            % ', '.join('%s=%d' % kv for kv in sorted(dropped_truncs.items(),
                                                      key=lambda kv: -kv[1])))
    return rows


def bounds(md, k):
    import grammar_v3 as G
    return G.TP_BOUNDS[md['trunc']] if k == 'tp' else G.BOUNDS[k]


def fit_joint(task, npz, traces, tgrid, rng):
    """Fit every mode's sampler together, with the modes REVEALED.

    One hill-climb over all modes' coordinates at once, because a single state runs through the
    whole trace and the modes share it: mode 1's parameters change the state mode 0 then sees.
    Fitting modes separately is wrong for exactly the long-dwell stretches that matter most.
    Starts from the prior midpoint, never from the truth.
    """
    pol = task['policy']
    cache = []
    for ti, steps in traces:
        Pm, _ = P.rebuild(npz['cal_lg'][ti][:len(steps)], npz['cal_tail'][ti][:len(steps)])
        cache.append((steps, Pm, npz['cal_ids'][ti], npz['cal_lg'][ti]))

    modes = []
    for m in range(pol['M']):
        md = dict(pol['modes'][m])
        for k in [x for x in KEYS if x in md]:
            lo, hi = bounds(md, k)
            md[k] = 0.5 * (lo + hi)
        modes.append(md)

    def nll(ms):
        tot, n = 0.0, 0
        for steps, Pm, ids, lg in cache:
            lp, _ = P.replay_trace(task, steps, Pm, ids, lg, tgrid, modes=ms)
            ok = np.isfinite(lp)
            tot -= float(lp[ok].sum())
            n += int(ok.sum())
        return tot / max(n, 1)

    coords = [(m, k) for m in range(pol['M']) for k in KEYS if k in modes[m]]
    best = nll(modes)
    for _ in range(PROPOSALS):
        m, k = coords[rng.randrange(len(coords))]
        cand = [dict(x) for x in modes]
        lo, hi = bounds(cand[m], k)
        cand[m][k] = float(np.clip(cand[m][k] + rng.gauss(0, 0.15 * (hi - lo)), lo, hi))
        v = nll(cand)
        if v < best:
            modes, best = cand, v
    return modes, best


def fit_guards(task, traces):
    """Recover the transition rules from the REVEALED mode sequence, by decision stumps.

    With modes given this is supervised and is the easy half of C1a; the sampler fit is the hard
    half.
    """
    pol = task['policy']
    if pol['M'] == 1:
        return []
    steps = [r for _, ss in traces for r in ss]
    guards = []
    for src in range(pol['M']):
        for dst in range(pol['M']):
            if src == dst:
                continue
            fired = [r for r in steps if r['mode'] == src and r['mode_after'] == dst]
            stay = [r for r in steps if r['mode'] == src and r['mode_after'] == src]
            if len(fired) < 5 or len(stay) < 5:
                continue
            best = None
            for sn in P.SENSORS:
                vals = sorted({round(r[sn], 4) for r in fired + stay})
                for thr in vals[::max(1, len(vals) // 40)]:
                    for cmp in ('ge', 'lt'):
                        ok = sum((r[sn] >= thr) if cmp == 'ge' else (r[sn] < thr)
                                 for r in fired) / len(fired)
                        bad = sum((r[sn] >= thr) if cmp == 'ge' else (r[sn] < thr)
                                  for r in stay) / len(stay)
                        if best is None or ok - bad > best[0]:
                            best = (ok - bad, sn, thr, cmp)
            if best and best[0] > 0.2:
                guards.append(dict(src=src, dst=dst, sensor=best[1], thr=best[2], cmp=best[3]))
    return guards


def predict_items(task, npz, fitted, tgrid):
    """Run the fitted machine forward over each prefix, then score the 8 candidates.

    The mode at the scored position comes from the FITTED guards, not the truth: C1a reveals
    modes on the calibration traces only.
    """
    last = npz['itm_lg'].shape[1] - 1
    out = []
    for n, it in enumerate(task['items']):
        steps = it['steps']
        Pm, _ = P.rebuild(npz['itm_lg'][n][:len(steps)], npz['itm_tail'][n][:len(steps)])
        st = P.fresh_state(fitted['modes'][0])
        cur, c = 0, 0
        for k, rec in enumerate(steps):
            md = fitted['modes'][cur]
            P.induced_at(task, rec, Pm[k], npz['itm_ids'][n][k], npz['itm_lg'][n][k],
                         md, st, tgrid)
            # advance unconditionally: mirostat's mu must track the generator even when the
            # emitted token is outside the stored support
            P.advance_state(st, md, task['policy']['gamma'], rec['lp_base'], rec['tok'])
            cur, c = P.step_fsm(fitted, cur, c, rec)
        Pl, _ = P.rebuild(npz['itm_lg'][n][last], npz['itm_tail'][n][last])
        ids = npz['itm_ids'][n][last]
        d = P.induced_at(task, it['scored'], Pl, ids, npz['itm_lg'][n][last],
                         fitted['modes'][cur], st, tgrid)
        v = np.array([float(d[np.where(ids == cid)[0][0]]) if np.any(ids == cid) else 0.0
                      for cid in it['candidates']])
        out.append(v / v.sum() if v.sum() > 0 else np.full(len(v), 1.0 / len(v)))
    return np.stack(out)


def sum_loss(Pm, task, eps, base):
    idx = np.array([it['true_index'] for it in task['items']])
    Q = (1 - eps) * Pm + eps * base
    return float(-np.log(np.clip(Q[np.arange(len(idx)), idx], 1e-7, 1.0)).sum()), len(idx)


def main():
    log('policy_cpu VERSION = %s' % getattr(P, 'VERSION', 'UNKNOWN'))
    env = json.load(open(os.path.join(SIM, 'env.json')))
    tgrid = np.array(env['tgrid'])
    rows = load(NTASK, tgrid)
    log('%d usable tasks | budget %s | <=%d fit steps | %d proposals'
        % (len(rows), BUDGET, FITSTEPS, PROPOSALS))

    tot = dict(base=0.0, ref=0.0, c1a=0.0, stateless=0.0)
    n_items, per_task, audit = 0, [], []
    for r in rows:
        npz = np.load(os.path.join(SIM, '%s.npz' % r['task_id']))
        traces = budget_traces(r, BUDGET, FITSTEPS)
        rng = random.Random(SEED + int(r['task_id'][1:]))
        audit.append(r['_audit'])

        b = np.stack([np.array(it['cand_lp_base']) for it in r['items']])
        base = np.exp(b - b.max(1, keepdims=True))
        base /= base.sum(1, keepdims=True)

        fitted = dict(r['policy'])
        fitted['modes'], _ = fit_joint(r, npz, traces, tgrid, rng)
        fitted['guards'] = fit_guards(r, traces)
        Pc = predict_items(r, npz, fitted, tgrid)

        # C3: the best SINGLE stateless sampler, same optimiser and same budget. The honest
        # stateless control; a one-parameter temperature fit is a far weaker baseline.
        flat_pol = dict(r['policy'], M=1, guards=[], dwell=[0], expiry=[10 ** 9],
                        modes=[dict(r['policy']['modes'][0])])
        flat_task = dict(r, policy=flat_pol)
        flat_tr = [(ti, [dict(x, mode=0) for x in ss]) for ti, ss in traces]
        fm, _ = fit_joint(flat_task, npz, flat_tr, tgrid, rng)
        flat_pol = dict(flat_pol, modes=fm)
        Ps = predict_items(dict(r, policy=flat_pol), npz, flat_pol, tgrid)

        lb = -float(np.sum([it['lp_base'] for it in r['items']]))
        lr = -float(np.sum([it['lp_policy'] for it in r['items']]))
        lc, n = sum_loss(Pc, r, EPS, base)
        ls, _ = sum_loss(Ps, r, EPS, base)
        tot['base'] += lb; tot['ref'] += lr; tot['c1a'] += lc; tot['stateless'] += ls
        n_items += n
        per_task.append(dict(task=r['task_id'], cond=r['condition'], n=n,
                             base=lb / n, ref=lr / n, c1a=lc / n, stateless=ls / n))
        if len(per_task) % 5 == 0:
            log('  %d/%d tasks' % (len(per_task), len(rows)))

    L = {k: v / n_items for k, v in tot.items()}
    head = L['base'] - L['ref']
    frac = lambda x: (L['base'] - x) / head

    print()
    print('  CPU-path audit: median |recomputed - stored| = %.2e nats' % float(np.median(audit)))
    print('  (above ~1e-3 nothing below is trustworthy)')
    print()
    print('  L_base %.4f | L_ref %.4f | headroom %.4f nats | %d items over %d tasks'
          % (L['base'], L['ref'], head, n_items, len(rows)))
    print()
    print('  %-34s %-9s %s' % ('control', 'L', 'of headroom'))
    print('  %-34s %-9.4f %.3f' % ('base model', L['base'], 0.0))
    print('  %-34s %-9.4f %.3f' % ('C3 best stateless fit', L['stateless'],
                                   frac(L['stateless'])))
    print('  %-34s %-9.4f %.3f' % ('C1a mode-oracle fit', L['c1a'], frac(L['c1a'])))
    print('  %-34s %-9.4f %.3f' % ('true policy (oracle)', L['ref'], 1.0))

    # per-condition breakdown: dwell=False is pointwise identifiable by design, so C1a should
    # be clearly higher there. If it is not, the fit is the problem rather than the observation.
    print()
    print('  by condition (C1a share of that cell\'s own headroom):')
    cells = {}
    for t in per_task:
        k = (t['cond']['M'], bool(t['cond']['dwell']))
        cells.setdefault(k, []).append(t)
    for k in sorted(cells):
        g = cells[k]
        hb = np.average([t['base'] for t in g], weights=[t['n'] for t in g])
        hr = np.average([t['ref'] for t in g], weights=[t['n'] for t in g])
        hc = np.average([t['c1a'] for t in g], weights=[t['n'] for t in g])
        sh = (hb - hc) / (hb - hr) if hb > hr else float('nan')
        print('    M=%d dwell=%-5s n=%-3d  %.3f' % (k[0], k[1], len(g), sh))

    c1a = frac(L['c1a'])
    print()
    if c1a >= THR_C1A:
        print('  GATE PASSED: C1a = %.3f >= %.2f' % (c1a, THR_C1A))
        print('  Build the predictor. It must reach >= 0.600 of headroom AND >= %.3f.'
              % (0.70 * c1a))
    else:
        print('  GATE FAILED: C1a = %.3f < %.2f' % (c1a, THR_C1A))
        print('  Revealing the true modes does not identify these policies from passive')
        print('  single-sample observation, so 0.600 is unreachable by construction and an')
        print('  amortized predictor cannot adjudicate anything. STOP HERE.')
        print('  Report the identifiability result, not a method failure.')

    json.dump(dict(L=L, headroom=head, c1a=c1a, stateless=frac(L['stateless']),
                   n_items=n_items, n_tasks=len(rows), audit_median=float(np.median(audit)),
                   threshold=THR_C1A, passed=bool(c1a >= THR_C1A), per_task=per_task),
              open(os.path.join(SIM, 'gate_c1a.json'), 'w'), indent=1)
    log('wrote %s/gate_c1a.json' % SIM)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /kaggle/working/planted.py
"""Planted-recovery test: can the fitter recover a policy it was handed the answer to?

    python planted.py

WHY THIS EXISTS. The C1a gate returned -1.911 of headroom -- worse than ignoring the policy
entirely -- and degraded monotonically with parameter count (M=1 +0.281, M=2 -0.4, M=3 -6.5).
That is the signature of a starved optimiser, not of an unidentifiable problem: 120 random
single-coordinate proposals over up to 24 dimensions barely leaves the prior midpoint. So the
gate measured the fitter, and its FAIL cannot be reported as a finding.

This test separates the two possibilities the gate confounded, on data where the answer is known:

    fit the policy from ITS OWN calibration traces, with the true modes revealed, and compare
    the fitted calibration NLL against the TRUE policy's calibration NLL on the same steps

    recovery gap ~ 0           the fitter works; the gate's failure was the optimiser
    recovery gap stays large   the data does not pin the parameters down even with modes given,
                               which IS the identifiability result

Reported per condition, because the gate's collapse was parameter-count-dependent and the
diagnosis has to be too.

Two optimisers are compared on identical budgets, since the point is to establish whether a
competent optimiser closes the gap:

    random      the gate's recipe: perturb one coordinate by a Gaussian, accept if better
    coordinate  sweep each coordinate over a line of values and take the best

All CPU.
"""
import os, json, math, time, random
import numpy as np

import policy_cpu as P

SIM = os.environ.get('SIM', './sim')
NTASK = int(os.environ.get('PLANT_NTASK', 12))
FITSTEPS = int(os.environ.get('FITSTEPS', 512))
BUDGET = os.environ.get('BUDGET', '8x128')
EVALS = int(os.environ.get('EVALS', 480))      # likelihood evaluations, identical per optimiser
LINE = int(os.environ.get('LINE', 6))          # values per coordinate for the line search
AUDIT_MAX = float(os.environ.get('AUDIT_MAX', 5e-3))
SEED = 71294
T0 = time.time()
KEYS = ('tp', 'eta', 't_lo', 't_hi', 't_exp', 'xtc_q', 'xtc_t', 'lam')


def log(m):
    print('[%7.1fs] %s' % (time.time() - T0, m), flush=True)


def bounds(md, k):
    import grammar_v3 as G
    return G.TP_BOUNDS[md['trunc']] if k == 'tp' else G.BOUNDS[k]


def budget_traces(task, spec, cap):
    n, ln = (int(x) for x in spec.split('x'))
    out, used = [], 0
    for ti in range(min(n, len(task['traces']))):
        steps = task['traces'][ti]['steps'][:ln]
        if used + len(steps) > cap and out:
            break
        out.append((ti, steps))
        used += len(steps)
    return out


def make_nll(task, npz, traces, tgrid):
    """Calibration NLL under given per-mode parameters, modes REVEALED."""
    cache = []
    for ti, steps in traces:
        Pm, _ = P.rebuild(npz['cal_lg'][ti][:len(steps)], npz['cal_tail'][ti][:len(steps)])
        cache.append((steps, Pm, npz['cal_ids'][ti], npz['cal_lg'][ti]))
    calls = [0]

    def nll(ms):
        calls[0] += 1
        tot, n = 0.0, 0
        for steps, Pm, ids, lg in cache:
            lp, _ = P.replay_trace(task, steps, Pm, ids, lg, tgrid, modes=ms)
            ok = np.isfinite(lp)
            tot -= float(lp[ok].sum())
            n += int(ok.sum())
        return tot / max(n, 1)
    return nll, calls


def start_mid(pol):
    """The gate's starting point: every coordinate at its prior midpoint."""
    ms = []
    for m in range(pol['M']):
        md = dict(pol['modes'][m])
        for k in [x for x in KEYS if x in md]:
            lo, hi = bounds(md, k)
            md[k] = 0.5 * (lo + hi)
        ms.append(md)
    return ms


def opt_random(nll, ms, coords, rng, evals):
    """The gate's optimiser: random single-coordinate Gaussian proposals."""
    best = nll(ms)
    for _ in range(evals - 1):
        m, k = coords[rng.randrange(len(coords))]
        cand = [dict(x) for x in ms]
        lo, hi = bounds(cand[m], k)
        cand[m][k] = float(np.clip(cand[m][k] + rng.gauss(0, 0.15 * (hi - lo)), lo, hi))
        v = nll(cand)
        if v < best:
            ms, best = cand, v
    return ms, best


def opt_coord(nll, ms, coords, rng, evals):
    """Coordinate-wise line search: sweep each coordinate, keep the best value, repeat.

    Same number of likelihood evaluations as the random optimiser. A coordinate sweep spends
    them where they inform a decision instead of taking random steps that are usually rejected.
    """
    best = nll(ms)
    used = 1
    while used + LINE <= evals:
        improved = False
        for (m, k) in coords:
            if used + LINE > evals:
                break
            lo, hi = bounds(ms[m], k)
            grid = np.linspace(lo, hi, LINE)
            vals = []
            for g in grid:
                cand = [dict(x) for x in ms]
                cand[m][k] = float(g)
                vals.append(nll(cand))
                used += 1
            j = int(np.argmin(vals))
            if vals[j] < best:
                ms = [dict(x) for x in ms]
                ms[m][k] = float(grid[j])
                best = vals[j]
                improved = True
        if not improved:
            break
    return ms, best


def main():
    log('policy_cpu VERSION = %s' % getattr(P, 'VERSION', 'UNKNOWN'))
    env = json.load(open(os.path.join(SIM, 'env.json')))
    tgrid = np.array(env['tgrid'])

    rows = []
    for line in open(os.path.join(SIM, 'tasks.jsonl'), encoding='utf-8'):
        r = json.loads(line)
        if r.get('degenerate') or not r['items']:
            continue
        npz = np.load(os.path.join(SIM, '%s.npz' % r['task_id']))
        if float(np.median(P.verify_against_generator(r, npz, tgrid, ntrace=1, n=128))) > AUDIT_MAX:
            continue
        rows.append(r)
        if len(rows) >= NTASK:
            break
    log('%d tasks | budget %s | <=%d steps | %d evaluations per optimiser'
        % (len(rows), BUDGET, FITSTEPS, EVALS))

    out = []
    for r in rows:
        npz = np.load(os.path.join(SIM, '%s.npz' % r['task_id']))
        traces = budget_traces(r, BUDGET, FITSTEPS)
        nll, calls = make_nll(r, npz, traces, tgrid)
        pol = r['policy']

        nll_true = nll(pol['modes'])          # the answer, for reference
        ms0 = start_mid(pol)
        nll_mid = nll(ms0)
        coords = [(m, k) for m in range(pol['M']) for k in KEYS if k in ms0[m]]

        _, nll_rand = opt_random(nll, [dict(x) for x in ms0], coords,
                                 random.Random(SEED + int(r['task_id'][1:])), EVALS)
        _, nll_coo = opt_coord(nll, [dict(x) for x in ms0], coords,
                               random.Random(SEED + int(r['task_id'][1:])), EVALS)

        # fraction of the midpoint-to-truth gap that the optimiser closed
        span = nll_mid - nll_true
        clos = lambda v: (nll_mid - v) / span if abs(span) > 1e-9 else float('nan')
        out.append(dict(task=r['task_id'], M=pol['M'], dwell=bool(r['condition']['dwell']),
                        ncoord=len(coords), true=nll_true, mid=nll_mid,
                        rand=nll_rand, coord=nll_coo,
                        clos_rand=clos(nll_rand), clos_coord=clos(nll_coo)))
        log('  %s M=%d coords=%-2d | NLL true %.4f mid %.4f | random %.4f (%.2f) '
            'coord %.4f (%.2f)'
            % (r['task_id'], pol['M'], len(coords), nll_true, nll_mid,
               nll_rand, out[-1]['clos_rand'], nll_coo, out[-1]['clos_coord']))

    print()
    print('  RECOVERY: fraction of the midpoint-to-truth NLL gap closed, by mode count')
    print('  %-6s %-5s %-10s %s' % ('modes', 'n', 'random', 'coordinate'))
    for M in sorted({o['M'] for o in out}):
        g = [o for o in out if o['M'] == M]
        print('  %-6d %-5d %-10.3f %.3f'
              % (M, len(g), float(np.median([o['clos_rand'] for o in g])),
                 float(np.median([o['clos_coord'] for o in g]))))
    print()
    print('  overall median: random %.3f | coordinate %.3f'
          % (float(np.median([o['clos_rand'] for o in out])),
             float(np.median([o['clos_coord'] for o in out]))))
    worse = sum(o['coord'] > o['mid'] for o in out)
    print('  tasks where the coordinate search ended worse than its start: %d of %d'
          % (worse, len(out)))

    print()
    med = float(np.median([o['clos_coord'] for o in out]))
    if med >= 0.80:
        print('  THE FITTER WORKS. A competent optimiser closes most of the gap, so the gate\'s')
        print('  -1.911 was the optimiser, not the problem. Re-run the gate with the')
        print('  coordinate search and read the result as a real measurement.')
    elif med >= 0.40:
        print('  PARTIAL. The optimiser is a large part of the gate\'s failure but not all of')
        print('  it. Re-run the gate with the coordinate search, and treat any residual')
        print('  shortfall as jointly attributable -- do not claim identifiability from it.')
    else:
        print('  THE DATA DOES NOT PIN THE PARAMETERS DOWN. Even with the true modes revealed')
        print('  and a competent optimiser, the calibration traces do not determine the')
        print('  policy. That IS the identifiability result, and it is the finding to report:')
        print('  passive single-sample observation is insufficient regardless of method.')
        print('  Check nll(true) vs nll(mid) first -- if that span is tiny, the likelihood is')
        print('  flat in the parameters and non-identifiability is direct.')

    json.dump(out, open(os.path.join(SIM, 'planted.json'), 'w'), indent=1)
    log('wrote %s/planted.json' % SIM)


if __name__ == '__main__':
    main()

In [ ]:
import os
os.makedirs('/kaggle/working/raw', exist_ok=True)
!cd /kaggle/working && OUT=/kaggle/working/raw python make_prompts.py
os.environ['PC'] = '/kaggle/working/raw/prompts_calibration.csv'
os.environ['PQ'] = '/kaggle/working/raw/prompts_query.csv'
import csv
for k in ('PC', 'PQ'):
    rows = list(csv.DictReader(open(os.environ[k], encoding='utf-8')))
    print(k, len(rows), 'prompts |', repr(rows[0]['text'])[:60])

## 3. Generate

Roughly 90 minutes and about 2.5 GB after compression. What is recorded per step: the base
model's top-512 distribution, the full-vocabulary statistics that truncation destroys
(`zstd`, and `logsumexp(z/T)` plus entropy on a temperature grid), the five sensors, the
ground-truth mode, transition provenance with signed threshold margins, the emitted token's
log-prob under **every** mode, inter-mode total variation, and the uniform draw the sampler
consumed.

Conditions are balanced across tasks: modes 1/2/3, dwell on/off, and thresholds placed at
designed quantiles of each sensor's measured distribution (well-bracketed at the 40-60th
percentile, poorly-bracketed at the 5th/95th). The margin condition is a two-pass design and
cannot be added afterwards.

Watch the summary at the end: if the degenerate count is high or the informationless fraction
is large, the run is weaker than it looks and the gate will inherit that.

In [ ]:
import os
os.environ.update(OUT='/kaggle/working/sim', NTASK='250', NTRACE='16', TRACELEN='256',
                  NITEM='40', TOPK='512', NREPLICATE='10')
!cd /kaggle/working && python -u simlab.py

## 4. The CPU path audit

Before any fit is trusted, check that the CPU reimplementation reproduces the GPU generator from
the stored distributions. Offline this was verified to a median total-variation gap of 5.6e-8
across all ten (order x truncation) combinations, after four bugs that each would have silently
corrupted every fit: renormalising the top-K, `top_nsigma` needing the full-vocabulary log-prob
std, `hf` order needing the tempered partition, and typical sampling needing the tempered
entropy. This cell confirms it holds on the real data.

In [ ]:
import json, glob, numpy as np, sys, importlib
sys.path.insert(0, '/kaggle/working')
import policy_cpu as P
P = importlib.reload(P)          # %%writefile changed the file; the module may still be cached
print('policy_cpu VERSION =', P.VERSION)
assert P.VERSION == '2026-10-09.mirostat-state-fix', (
    'STALE MODULE: re-run the policy_cpu writefile cell, then this one again')
env = json.load(open('/kaggle/working/sim/env.json'))
rows = [json.loads(l) for l in open('/kaggle/working/sim/tasks.jsonl', encoding='utf-8')][:8]
errs = []
for r in rows:
    npz = np.load('/kaggle/working/sim/%s.npz' % r['task_id'])
    e = P.verify_against_generator(r, npz, np.array(env['tgrid']))
    errs.append(np.median(e))
    tr = ','.join(m['trunc'] for m in r['policy']['modes'])
    kept = np.mean([s.get('kept', np.nan) for t in r['traces'][:2] for s in t['steps'][:256]])
    # how often the emitted token falls outside the stored top-512: those steps have no
    # likelihood, and (before the fix) also skipped the state advance
    oos = []
    for ti in range(min(2, len(r['traces']))):
        ids = npz['cal_ids'][ti]
        for k, st_ in enumerate(r['traces'][ti]['steps'][:256]):
            oos.append(not np.any(ids[k] == st_['tok']))
    print('%s  median %.2e  max %.2e | order=%-6s truncs=%-22s kept=%.3f oos=%.3f'
          % (r['task_id'], np.median(e), e.max(), r['policy']['order'], tr, kept,
             float(np.mean(oos))))
print()
print('overall median %.2e' % np.median(errs))
print('under ~1e-3 the CPU path is sound; above that, nothing downstream is trustworthy')
print()
print('if failures concentrate on one truncation family, that sampler is not reconstructible')
print('from the stored top-512 and those tasks should be excluded rather than regenerated')

## 4b. Backfill the scored position (only needed for a run generated before this was stored)

The first version of the generator stored the base distribution for the 24 prefix steps but not
for the scored position itself. Truncation samplers need the whole distribution to decide what
survives, so the step being predicted was uncomputable offline. `simlab.py` now stores it; this
cell backfills a run that predates the fix, with one short forward pass per item instead of
regenerating. It is idempotent and skips tasks that already have it.

In [ ]:
import os
os.environ['SIM'] = '/kaggle/working/sim'
!cd /kaggle/working && python -u patch_scored.py

## 5. The gate

Fits each mode's sampler on the calibration steps **labelled with that mode**, recovers the
transition rules from the revealed mode sequence, then runs the fitted machine forward over each
scored prefix. Reported against the base model (0) and the true policy (1), alongside `C3`, the
best single stateless sampler fitted with the same optimiser and budget — which is the honest
stateless control, not the one-parameter temperature fit used previously.

CPU only. About 15 minutes.

In [ ]:
import os
os.environ.update(SIM='/kaggle/working/sim', GATE_NTASK='40', FITSTEPS='1024',
                  PROPOSALS='120', BUDGET='16x256', EPS='0.03')
!cd /kaggle/working && python -u gate_c1a.py

## 6. Verdict

The cell below restates the gate's decision and, if it failed, says plainly what the result is.

A failure here is not a wasted run. "Even with the modes revealed, passive single-sample
observation does not identify these policies" is a sharper and more defensible claim than a
method comparison, and it is supported by everything this run recorded: the inter-mode total
variation per step bounds what any mode inference could achieve, and the guard margins bound how
tightly each threshold is bracketed by the observed trajectory.

In [ ]:
import json
g = json.load(open('/kaggle/working/sim/gate_c1a.json'))
print('C1a                %.3f of headroom' % g['c1a'])
print('C3 stateless       %.3f' % g['stateless'])
print('headroom           %.4f nats over %d items, %d tasks'
      % (g['headroom'], g['n_items'], g['n_tasks']))
print('CPU audit median   %.2e nats' % g['audit_median'])
print()
if g['passed']:
    print('PASS -- build the amortized mode predictor.')
    print('   it must reach >= 0.600 of headroom AND >= %.3f' % (0.70 * g['c1a']))
else:
    print('FAIL -- stop. Report the identifiability result:')
    print('   revealing the true modes reaches only %.3f of headroom, so mode inference')
    print('   is not the bottleneck and no amortized predictor can clear 0.600.')

## 7. Package the simulation

Save the run as a dataset so every later analysis is CPU-only and nothing needs regenerating.

In [ ]:
!du -sh /kaggle/working/sim
!ls /kaggle/working/sim | head -5
!cat /kaggle/working/sim/env.json